# PI Questions — Phase 0–2 Follow-up Analysis

Working notebook for the questions the PI raised (meeting 2026-08-18) about the
shearing-box + multigrid self-gravity program, answered by **further analysis of the
existing test data** (`validation/run`, `validation/run_epicycle`, and the two 256²
collapse archives). Companion documents: `multigrid_selfgravity_validation.pdf`
(the results being interrogated) and `multigrid_validation_tests.ipynb` (the notebook
that generated the caches).

Conventions, shared with the validation notebook:
- stored **unexecuted**; cells read caches and never launch simulations unless a
  question explicitly requires a new run (none so far),
- one section per phase, filled in as each question is answered: the question as
  posed, the data used, the method, the result, and the takeaway.


In [2]:
const REPO   = expanduser("~/Library/CloudStorage/Dropbox/Research/code/athenak-multigrid")
const VAL    = joinpath(REPO, "validation")
const RUN    = joinpath(VAL, "run")           # caches written by multigrid_validation_tests.ipynb
const RUNEPI = joinpath(VAL, "run_epicycle")  # 20-orbit epicycle caches
const D256MG  = joinpath(RUN, "swing_mg_uniform_256_tlim14")   # NAS Pleiades, 64 ranks, multigrid
const D256FFT = joinpath(RUN, "swing_fft_uniform_256_tlim14")  # laptop serial, FFT solver
for d in (RUN, RUNEPI, D256MG, D256FFT)
    isdir(d) || error("cache directory $d missing — this notebook only analyzes existing data")
end
using CairoMakie, Printf, Statistics
CairoMakie.activate!(type="png")
set_theme!(Theme(fontsize=13, Axis=(xgridcolor=(:gray, 0.25), ygridcolor=(:gray, 0.25))))
const C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100";

In [3]:
## Readers — copied verbatim from multigrid_validation_tests.ipynb so both notebooks
## parse the caches identically; plus small helpers specific to this notebook.

"Read an AthenaK .hst file into a matrix (rows = outputs)."
function read_hst(f)
    rows = Float64[]; ncol = 0
    for ln in eachline(f)
        startswith(strip(ln), "#") && continue
        v = parse.(Float64, split(ln)); ncol = length(v); append!(rows, v)
    end
    permutedims(reshape(rows, ncol, :))
end

"Read an AthenaK .bin file into per-MeshBlock records (multilevel-safe)."
function read_bin_blocks(filename)
    open(filename, "r") do io
        startswith(readline(io), "Athena binary output") || error("not an AthenaK bin")
        npre = parse(Int, split(readline(io), "=")[end])
        ph = Dict(String(strip(k)) => String(strip(v)) for (k, v) in
                  (split(readline(io), "=") for _ in 1:npre-1))
        locsize = parse(Int, ph["size of location"])
        varsize = parse(Int, ph["size of variable"])
        nvars = parse(Int, split(readline(io), "=")[end])
        vars = String.(split(readline(io))[2:end])
        hsize = parse(Int, split(readline(io), "=")[end])
        header = String(read(io, hsize))
        m = match(r"nghost\s*=\s*(\d+)", header)
        ng = parse(Int, m.captures[1])
        locT = locsize == 8 ? Float64 : Float32
        varT = varsize == 8 ? Float64 : Float32
        blocks = NamedTuple[]
        while !eof(io)
            idx = Int.(reinterpret(Int32, read(io, 24))) .- ng
            n1 = idx[2]-idx[1]+1; n2 = idx[4]-idx[3]+1; n3 = idx[6]-idx[5]+1
            logical = Int.(reinterpret(Int32, read(io, 16)))       # lx1,lx2,lx3,level
            geom = Float64.(reinterpret(locT, read(io, 6*locsize)))  # x1min..x3max
            raw = reinterpret(varT, read(io, n1*n2*n3*nvars*varsize))
            data = reshape(Float64.(raw), (n1, n2, n3, nvars))
            push!(blocks, (logical=logical, geom=geom, data=data))
        end
        (blocks=blocks, vars=vars)
    end
end

"Keep the last monotone-time segment (guards against appended reruns)."
seg(h) = begin
    s = 1
    for i in 2:size(h,1); h[i,1] <= h[i-1,1] && (s = i); end
    h[s:end, :]
end

"Read a whitespace-separated numeric table (e.g. the rhomax series); '#' lines skipped."
readcols(f) = begin
    rows = [parse.(Float64, split(l)) for l in eachline(f)
            if !isempty(strip(l)) && !startswith(strip(l), "#")]
    permutedims(hcat(rows...))
end

"Mosaic one variable of a UNIFORM-mesh bin into a dense 3D array (grid-index order)."
function mosaic(filename, var)
    fb = read_bin_blocks(filename)
    iv = findfirst(==(var), fb.vars)
    iv === nothing && error("variable $var not in $(fb.vars)")
    b1 = fb.blocks[1]
    n1, n2, n3 = size(b1.data)[1:3]
    nb1 = maximum(b.logical[1] for b in fb.blocks) + 1
    nb2 = maximum(b.logical[2] for b in fb.blocks) + 1
    nb3 = maximum(b.logical[3] for b in fb.blocks) + 1
    all(b.logical[4] == fb.blocks[1].logical[4] for b in fb.blocks) ||
        error("mosaic() is for uniform meshes only")
    A = zeros(n1*nb1, n2*nb2, n3*nb3)
    for b in fb.blocks
        o1, o2, o3 = b.logical[1]*n1, b.logical[2]*n2, b.logical[3]*n3
        A[o1+1:o1+n1, o2+1:o2+n2, o3+1:o3+n3] .= b.data[:, :, :, iv]
    end
    A
end;

## Phase 0 — refinement + FARGO in the shearing box


### Q0.1 — Do density amplitudes change across the fine–coarse interfaces in the epicycle runs?

*(as posed at the 2026-08-18 meeting; data: `epi_ring`, `epi_bndry`)*

**Why the epicycle is the sharpest probe.** `ipert=1` initializes a spatially **uniform**
epicycle $v_x(0)=A$: the exact nonlinear solution keeps every field spatially uniform for
all time — in particular $\rho \equiv \rho_0$. There is no physical density amplitude at
all, so *any* nonzero $\delta\rho = \rho - \rho_0$, and especially structure localized at
the refinement boundaries, is numerical — injected by prolongation/restriction, the
interface flux corrections, the per-level FARGO shift, or the level-1 shear wrap.

**Data.** The 20-orbit endurance runs (`validation/run_epicycle`, 202 float32 snapshots
each, output cadence 0.628):

| run | mesh | fine–coarse interfaces |
|---|---|---|
| `epi_unif` | $128\times128\times4$ uniform | none (control) |
| `epi_ring` | level-1 ring $|x|<5$ | $x = \pm 5$ |
| `epi_bndry` | level-1 annuli $|x|>5$ | $x = \pm 5$, refined shear wrap at $x=\pm10$ |

**Method.** Sweep every cell of every block of every snapshot: global
$\max|\rho - 1|$ per run and per snapshot, plus the spatial spreads
($\max - \min$) of all other primitives. Float32 caveat: the snapshot quantum at
$\rho = 1$ is $\sim 6\times10^{-8}$; the statement below is at that resolution.


#### Derivation — the epicycle equations of motion and the reference solution

**1. The shearing-sheet frame and its forces.**
Work in a local Cartesian patch co-rotating with the disk at radius $R_0$ with angular
frequency $\Omega_0 = \Omega(R_0)$: $x = R - R_0$ (radial), $y = R_0(\phi - \Omega_0 t)$
(azimuthal), $z$ vertical. In this rotating frame two inertial/gravity terms survive at
first order in $x/R_0$:

*Tidal term.* The effective potential is $\Phi_{\rm eff}(R)=\Phi(R)-\tfrac12\Omega_0^2R^2$.
Using radial force balance of circular orbits, $\mathrm{d}\Phi/\mathrm{d}R = R\,\Omega^2(R)$,
and the local power law $\Omega \propto R^{-q}$ with $q \equiv -\mathrm{d}\ln\Omega/\mathrm{d}\ln R$,
so that $\Omega^2(R_0{+}x) \simeq \Omega_0^2(1 - 2q\,x/R_0)$:
$$
\frac{\mathrm{d}\Phi_{\rm eff}}{\mathrm{d}R}\Big|_{R_0+x}
= (R_0{+}x)\,\Omega^2(R_0{+}x) - \Omega_0^2 (R_0{+}x)
= -2q\,\Omega_0^2\,x + O(x^2/R_0),
$$
i.e. a radial force $+2q\Omega^2 x\,\hat{\mathbf{x}}$ (outward beyond $R_0$, inward inside —
the tidal stretching). This expansion **is** the linearization that defines the sheet;
everything after it is exact within the sheet.

*Coriolis term.* $-2\boldsymbol{\Omega}\times\mathbf{v}
= 2\Omega v_y\,\hat{\mathbf{x}} - 2\Omega v_x\,\hat{\mathbf{y}}$ (with
$\boldsymbol{\Omega} = \Omega\hat{\mathbf{z}}$; drop the subscript 0 from here on).

The momentum equations of the sheet (unstratified, no vertical gravity) are then
$$
\frac{\mathrm{D} v_x}{\mathrm{D}t} = 2\Omega v_y + 2q\Omega^2 x
- \frac{1}{\rho}\partial_x p, \qquad
\frac{\mathrm{D} v_y}{\mathrm{D}t} = -2\Omega v_x - \frac{1}{\rho}\partial_y p .
$$

**2. Equilibrium.** $v_x = 0$, $v_y = -q\Omega x$ (uniform $\rho,p$) balances exactly:
$2\Omega(-q\Omega x) + 2q\Omega^2 x = 0$. This is the background shear flow the code's
orbital-advection (FARGO) mode splits off and advects analytically; the evolved azimuthal
variable is the fluctuation $v_y' \equiv v_y + q\Omega x$.

**3. Spatially uniform perturbation — an exact reduction.** Take
$v_x = v_x(t)$, $v_y' = v_y'(t)$ uniform in space (the `ipert=1` initial condition:
$v_x(0)=A$, $v_y'(0)=0$), with $\rho$, $p$ uniform. Then:
- pressure gradients vanish;
- the fluctuation self-advection $\mathbf{v}'\!\cdot\!\nabla\mathbf{v}'$ vanishes
  identically (nothing depends on position) — **no linearization in the amplitude $A$
  is needed**; the only surviving advection is $v_x\,\partial_x$ acting on the
  background $-q\Omega x$;
- continuity gives $\partial_t\rho = -\nabla\!\cdot\!(\rho\mathbf{v}) = 0$: the density
  stays **exactly** $\rho_0$ (this is what Q0.1 exploits).

The $x$-equation, with $v_y = -q\Omega x + v_y'$:
$$
\dot v_x = 2\Omega(-q\Omega x + v_y') + 2q\Omega^2 x = 2\Omega\,v_y' .
$$
The $y$-equation, keeping the advection of the background shear
($v_x\,\partial_x v_y = -q\Omega\,v_x$):
$$
\dot v_y' - q\Omega\,v_x = -2\Omega\,v_x
\quad\Longrightarrow\quad
\dot v_y' = -(2-q)\,\Omega\,v_x .
$$

**4. The epicyclic oscillator and the reference solution.** Combining,
$$
\ddot v_x = -\,2(2-q)\,\Omega^2\, v_x \equiv -\kappa^2 v_x,
\qquad
\boxed{\;\kappa^2 = 2(2-q)\,\Omega^2\;}
$$
which is the local form of the general epicyclic frequency
$\kappa^2 = \frac{1}{R^3}\frac{\mathrm{d}}{\mathrm{d}R}\big(R^4\Omega^2\big)
= 4\Omega^2 + 2R\Omega\,\Omega' = (4-2q)\,\Omega^2$. For Keplerian $q = 3/2$:
$\kappa = \Omega$ (the notebook's `κ = sqrt(2*(2-q))*Ω`, $=1$ at $\Omega=1$).
With the code's initial condition:
$$
v_x(t) = A\cos\kappa t, \qquad
v_y'(t) = \frac{\dot v_x}{2\Omega} = -\frac{A\kappa}{2\Omega}\,\sin\kappa t .
$$
The orbit in velocity space is an ellipse with axis ratio $\kappa/2\Omega$ ($=1/2$ at
$q=3/2$), traversed clockwise; the corresponding displacement is
$x(t) = x_0 + (A/\kappa)\sin\kappa t$ — the classical retrograde epicycle.

**5. The two diagnostics the notebooks compare against.**
- *Direct trajectory:* volume-averaged $\langle\rho v_x\rangle/\langle\rho\rangle$ vs
  $A\cos\kappa t$ (and $v_y'$ vs its sine), from the float64 history files. The
  20-orbit envelope errors (0.6–2.4% of $A$) are RK2 phase drift, not amplitude error.
- *Amplitude invariant:* eliminating $t$ from the solution,
$$
\mathcal{A}(t) \equiv \sqrt{v_x^2 + \left(\frac{2\Omega}{\kappa}\right)^2 v_y'^2} = A
= {\rm const},
$$
which is conserved by the exact dynamics for **any** amplitude, so its drift isolates
numerical damping/growth from phase error (measured: $+0.008\%$ ring / $+0.064\%$
uniform over 20 orbits — RK2's slow secular growth on an oscillator).
- For the short `ipert=1` runs the history comparison uses the volume-integrated
  $x$-kinetic energy instead: $\int\tfrac12\rho v_x^2\,\mathrm{d}V
  = \tfrac12\rho_0 A^2\cos^2(\kappa t)\,V_{\rm box}$ (hence the `0.5*V*amp^2*cos(t)^2`
  with $V_{\rm box} = 0.125$ for the $1\times1\times0.125$ shwave box).

**6. Why this makes the interface test sharp (recap).** The solution is uniform and
exact at finite amplitude, so the *only* error channels are time integration (spatially
uniform, harmless to Q0.1) and any operation that fails to preserve spatial constants —
which is precisely what the refinement/FARGO/shear-wrap machinery is being interrogated
for. A defect appears as $\delta\rho \ne 0$ or velocity spread at the interface;
Q0.1 found exactly zero at float32 resolution.


In [4]:
## Q0.1 — density amplitudes across fine-coarse interfaces in the epicycle runs.
## The ipert=1 epicycle is SPATIALLY UNIFORM: the exact solution keeps rho = rho0
## everywhere, so any spatial density structure is numerical, and the fine-coarse
## interfaces (x = +-5 in both meshes) are where it would enter. Sweep every
## snapshot of the three 20-orbit runs (~2 min at first pass; caches to
## run_epicycle/uniformity_epi.txt): global max |rho - 1| plus the spatial spreads
## of the other primitives (float32 snapshots: quantum ~6e-8 on rho = 1).
function uniformity_sweep()
    cf = joinpath(RUNEPI, "uniformity_epi.txt")
    if !isfile(cf)
        open(cf, "w") do io
            println(io, "# run  t  max|dens-1|  spread(velx)  spread(vely)  max|velz|  spread(eint)")
            for bn in ("epi_unif", "epi_ring", "epi_bndry")
                fs = sort(filter(f -> startswith(f, bn * ".hydro_w"),
                                 readdir(joinpath(RUNEPI, "bin"))))
                for f in fs
                    t = 0.628 * parse(Int, split(f, '.')[end-1])
                    fb = read_bin_blocks(joinpath(RUNEPI, "bin", f))
                    iv = Dict(v => k for (k, v) in enumerate(fb.vars))
                    g(v, red) = red(red(b.data[:, :, :, iv[v]]) for b in fb.blocks)
                    sprd(v) = g(v, maximum) - g(v, minimum)
                    println(io, join([bn; @sprintf("%.4f", t);
                                      [@sprintf("%.6e", x) for x in
                                       (maximum(maximum(abs.(b.data[:, :, :, iv["dens"]] .- 1.0))
                                                for b in fb.blocks),
                                        sprd("velx"), sprd("vely"),
                                        maximum(maximum(abs.(b.data[:, :, :, iv["velz"]]))
                                                for b in fb.blocks),
                                        sprd("eint"))]], "  "))
                end
                flush(io)
            end
        end
    end
    tab = Dict{String,Matrix{Float64}}()
    for bn in ("epi_unif", "epi_ring", "epi_bndry")
        rows = [parse.(Float64, split(l)[2:end]) for l in eachline(cf)
                if startswith(l, bn)]
        tab[bn] = permutedims(hcat(rows...))
    end
    tab
end
uni = uniformity_sweep()
println("worst case over all 202 snapshots x 20 orbits (float32 quantum on rho=1: 6e-8):")
@printf("%-10s %14s %14s %14s %12s %14s\n",
        "run", "max|dens-1|", "spread(velx)", "spread(vely)", "max|velz|", "spread(eint)")
for bn in ("epi_unif", "epi_ring", "epi_bndry")
    m = uni[bn]
    @printf("%-10s %14.3e %14.3e %14.3e %12.3e %14.3e\n", bn,
            maximum(m[:, 2]), maximum(m[:, 3]), maximum(m[:, 4]),
            maximum(m[:, 5]), maximum(m[:, 6]))
end

worst case over all 202 snapshots x 20 orbits (float32 quantum on rho=1: 6e-8):
run           max|dens-1|   spread(velx)   spread(vely)    max|velz|   spread(eint)
epi_unif        0.000e+00      0.000e+00      0.000e+00    0.000e+00      0.000e+00
epi_ring        0.000e+00      0.000e+00      0.000e+00    0.000e+00      0.000e+00
epi_bndry       0.000e+00      0.000e+00      0.000e+00    0.000e+00      0.000e+00


In [5]:
## Guard against a degenerate read: the (spatially uniform) velx value must still
## OSCILLATE — it has to track the analytic epicycle A cos(kappa*t) snapshot by
## snapshot. The growing diff is the documented RK2 phase/amplitude drift of the
## 20-orbit runs (time-integration error, spatially blind).
"exact snapshot time from the bin header"
bin_time(f) = open(f) do io
    for _ in 1:40
        m = match(r"time=([0-9eE+.-]+)", readline(io))
        m === nothing || return parse(Float64, m.captures[1])
    end
    error("no time= in header of $f")
end
for (bn, idx) in (("epi_ring", 0), ("epi_ring", 25), ("epi_bndry", 100), ("epi_bndry", 201))
    f = joinpath(RUNEPI, "bin", bn * @sprintf(".hydro_w.%05d.bin", idx))
    t = bin_time(f)
    fb = read_bin_blocks(f)
    iv = findfirst(==("velx"), fb.vars)
    v = fb.blocks[1].data[1, 1, 1, iv]
    @printf("%-9s t=%8.3f: velx = %+.7f   0.1 cos(t) = %+.7f   diff %.1e\n",
            bn, t, v, 0.1 * cos(t), abs(v - 0.1 * cos(t)))
end

epi_ring  t=   0.000: velx = +0.1000000   0.1 cos(t) = +0.1000000   diff 1.5e-09
epi_ring  t=  15.709: velx = -0.1000008   0.1 cos(t) = -0.0999999   diff 8.9e-07
epi_bndry t=  62.802: velx = +0.0999692   0.1 cos(t) = +0.0999565   diff 1.3e-05
epi_bndry t= 125.664: velx = +0.1000061   0.1 cos(t) = +0.1000000   diff 6.1e-06


**Result — no: there is no density amplitude anywhere, at any time.**
$\max|\rho - 1| = 0.0$ — exactly zero, not one float32 ulp — in every cell of all 202
snapshots of all three meshes, and every other primitive is spatially uniform to the bit
(while the uniform $v_x$ value tracks $A\cos\kappa t$, so the fields are live, not a
read artifact). Since the global maximum is exactly zero, the per-column profile is
identically zero: **the fine–coarse interfaces at $x=\pm5$ — and the level-1
shear-periodic wrap at $x=\pm10$ in `epi_bndry` — inject no density perturbation over
20 orbits.**

**Mechanism.** For a spatially uniform state every operator in the loop is
constant-preserving: PLM reconstruction/fluxes of a constant, interface flux corrections
between equal fluxes, prolongation/restriction of constants, the per-level conservative
FARGO shift, and the shear-periodic wrap of a uniform field. Any indexing or conservation
defect in the refinement machinery would appear as at least an ulp-level breach; none does.

**Caveats.** Snapshots are float32; float64 round-off structure below $6\times10^{-8}$
is invisible here. The float64 history diagnostics corroborate: mass drift exactly 0,
amplitude-invariant drift $+0.008\%$ (ring) / $+0.064\%$ (uniform) over 20 orbits —
a time-integration effect common to all meshes, not an interface one.

#### Extension — 100-orbit reruns (requested 2026-08-19)

The three endurance runs extended to $t_{\rm lim} = 200\pi$ (100 orbits, $5\times$ the
documented runs), with new basenames so the 20-orbit caches stay untouched, and the
**bin** cadence thinned $\times10$ (the history cadence is unchanged):

```
ATH=build/src/athena   # run in validation/run_epicycle
$ATH -i inputs/shearing_box/epicycle.athinput          job/basename=epi_unif100  time/tlim=628.3185307179587 output2/dt=6.28
$ATH -i inputs/shearing_box/epicycle_smr_ring.athinput  job/basename=epi_ring100  time/tlim=628.3185307179587 output2/dt=6.28
$ATH -i inputs/shearing_box/epicycle_smr_bndry.athinput job/basename=epi_bndry100 time/tlim=628.3185307179587 output2/dt=6.28
```

Wall times (this laptop, serial, three concurrent): uniform 7 min, each SMR run 69 min
(the fine level carries $4\times$ the replaced cells and halves the global timestep:
$\Delta t = 0.0175$ vs $0.034$).


In [ ]:
## Q0.1 extension — the 100-orbit reruns (tlim = 200π, bin cadence thinned to 6.28):
## vx/vx(0) for the three runs against the analytic epicycle A cos(κt).
## Top: full 100 orbits (individual oscillations blur into the ±1 band; what is
## visible is the envelope). Bottom: the last two orbits, where the accumulated RK2
## phase drift separates the runs from the analytic curve and from each other.
amp, Ω, q = 0.1, 1.0, 1.5
κ = sqrt(2 * (2 - q)) * Ω
Torb = 2π / Ω
runs100 = (("uniform 128²", "epi_unif100", C1, :circle),
           ("ring SMR", "epi_ring100", C3, :rect),
           ("bndry SMR", "epi_bndry100", C4, :diamond))
H100 = Dict(bn => read_hst(joinpath(RUNEPI, bn * ".hydro.hst")) for (_, bn, _, _) in runs100)

fig = Figure(size=(980, 640))
ax1 = Axis(fig[1, 1]; xlabel="t / T_orb", ylabel="vₓ / vₓ(0)",
           title="epicycle vₓ over 100 orbits — three meshes vs analytic cos(κt)")
hlines!(ax1, [-1, 1]; color=(:gray, 0.5), linestyle=:dash)
td = range(0, 100Torb; length=20001)
lines!(ax1, td ./ Torb, cos.(κ .* td); color=(C2, 0.35), linewidth=0.4, label="analytic")
for (nm, bn, c, _) in runs100
    h = H100[bn]
    lines!(ax1, h[:, 1] ./ Torb, (h[:, 4] ./ h[:, 3]) ./ amp; color=(c, 0.8),
           linewidth=0.5, label=nm)
end
axislegend(ax1; position=:rt, framevisible=false, orientation=:horizontal)
ylims!(ax1, -1.6, 1.6)

ax2 = Axis(fig[2, 1]; xlabel="t / T_orb", ylabel="vₓ / vₓ(0)", height=190,
           title="last two orbits: accumulated phase drift vs the analytic solution")
tz = range(98Torb, 100Torb; length=1200)
lines!(ax2, tz ./ Torb, cos.(κ .* tz); color=C2, linewidth=2.5, label="analytic")
for (nm, bn, c, mk) in runs100
    h = H100[bn]
    s = h[:, 1] ./ Torb .>= 98
    scatterlines!(ax2, h[s, 1] ./ Torb, (h[s, 4] ./ h[s, 3]) ./ amp;
                  color=c, marker=mk, markersize=6, linewidth=1, label=nm)
end
axislegend(ax2; position=:rb, framevisible=false, orientation=:horizontal, labelsize=10)
fig

In [ ]:
## Q0.1 extension (continued) — drift metrics at 100 orbits, and the interface
## uniformity sweep repeated on the 100-orbit snapshots (cadence 6.28, 102 each;
## caches to run_epicycle/uniformity_epi100.txt).
println("run           vx range/amp            invariant drift   max|vx-analytic|/amp")
for (nm, bn, _, _) in runs100
    h = H100[bn]
    vx = h[:, 4] ./ h[:, 3]; vy = h[:, 5] ./ h[:, 3]
    A = sqrt.(vx .^ 2 .+ (2Ω / κ)^2 .* vy .^ 2)
    @printf("%-12s [%+.5f, %+.5f]   %+.3f%%           %.4f\n", nm,
            minimum(vx) / amp, maximum(vx) / amp, 100 * (A[end] - A[1]) / amp,
            maximum(abs.(vx .- amp .* cos.(κ .* h[:, 1]))) / amp)
end
n = min(size(H100["epi_ring100"], 1), size(H100["epi_bndry100"], 1))
@printf("\nbndry vs ring, all history columns: max diff = %.1e (20-orbit runs were bit-identical)\n",
        maximum(abs.(H100["epi_ring100"][1:n, :] .- H100["epi_bndry100"][1:n, :])))

function uniformity_sweep100()
    cf = joinpath(RUNEPI, "uniformity_epi100.txt")
    if !isfile(cf)
        open(cf, "w") do io
            println(io, "# run  t  max|dens-1|  spread(velx)  spread(vely)  max|velz|  spread(eint)")
            for (_, bn, _, _) in runs100
                fs = sort(filter(f -> startswith(f, bn * ".hydro_w"),
                                 readdir(joinpath(RUNEPI, "bin"))))
                for f in fs
                    fb = read_bin_blocks(joinpath(RUNEPI, "bin", f))
                    iv = Dict(v => k for (k, v) in enumerate(fb.vars))
                    g(v, red) = red(red(b.data[:, :, :, iv[v]]) for b in fb.blocks)
                    sprd(v) = g(v, maximum) - g(v, minimum)
                    println(io, join([bn; @sprintf("%.4f", bin_time(joinpath(RUNEPI, "bin", f)));
                                      [@sprintf("%.6e", x) for x in
                                       (maximum(maximum(abs.(b.data[:, :, :, iv["dens"]] .- 1.0))
                                                for b in fb.blocks),
                                        sprd("velx"), sprd("vely"),
                                        maximum(maximum(abs.(b.data[:, :, :, iv["velz"]]))
                                                for b in fb.blocks),
                                        sprd("eint"))]], "  "))
                end
                flush(io)
            end
        end
    end
    cf
end
cf100 = uniformity_sweep100()
println("\nworst case over the 100-orbit snapshots (float32 quantum 6e-8):")
@printf("%-14s %14s %14s %14s %12s %14s\n",
        "run", "max|dens-1|", "spread(velx)", "spread(vely)", "max|velz|", "spread(eint)")
for (_, bn, _, _) in runs100
    rows = [parse.(Float64, split(l)[2:end]) for l in eachline(cf100) if startswith(l, bn)]
    m = permutedims(hcat(rows...))
    @printf("%-14s %14.3e %14.3e %14.3e %12.3e %14.3e\n", bn,
            maximum(m[:, 2]), maximum(m[:, 3]), maximum(m[:, 4]),
            maximum(m[:, 5]), maximum(m[:, 6]))
end

**100-orbit results.**

| run | $v_x$ range / $A$ | invariant drift | max $|v_x - A\cos\kappa t|/A$ |
|---|---|---|---|
| uniform $128^2$ | $[-1.00059, +1.00045]$ | $+0.321\%$ | $0.124$ |
| ring SMR | $[-1.00001, +1.00000]$ | $+0.040\%$ | $0.031$ |
| bndry SMR | $[-1.00001, +1.00000]$ | $+0.040\%$ | $0.031$ |

- **Q0.1 extends unchanged to 100 orbits**: the uniformity sweep over all
  $3\times102$ new snapshots again gives $\max|\rho-1| = 0.0$ exactly, and every
  primitive spatially uniform to the bit — five times the documented duration, still
  not one float32 ulp of interface noise.
- **`epi_ring100` and `epi_bndry100` are bit-identical to each other through 100
  orbits** (max difference over all history columns exactly $0.0$), extending the
  documented 20-orbit bit-identity: for a uniform field the two refined layouts are
  the same computation.
- **The error budget is pure RK2 time integration, and it scales like it.** Both SMR
  runs integrate the whole domain at the fine-level $\Delta t$ (global timestep), half
  the uniform run's. Measured uniform/SMR ratios: phase error $0.124/0.031 = 4.0 =
  (\Delta t_u/\Delta t_s)^2$ (RK2 phase error $\propto \Delta t^2$ per unit time) and
  invariant drift $0.321/0.040 = 8.0 = (\Delta t_u/\Delta t_s)^3$ (RK2 amplitude
  growth $\propto \Delta t^3$ per unit time). The refined runs are *more* accurate
  purely because they take smaller steps — the interfaces contribute nothing, which
  is the 100-orbit restatement of Q0.1's answer.
- At 100 orbits the uniform run's phase lead is visible by eye in the zoom panel
  ($\sim$1/8 of a period) while its envelope stays within $6\times10^{-4}$ of $\pm1$ —
  the standard oscillator picture: RK2's phase error accumulates linearly in $t$,
  amplitude error much more slowly.


### Q0.2 — Why do the density amplitudes flicker between frames 49 and 50 of the shwave comparison movie?

*(as posed at the 2026-08-18 meeting, about
`movie_runs/dens_compare_shw_uniform_vs_shw_smr_ring2_vs_shw_smr_bndry2_vs_shw_amr_ring_grid.mp4`:
between frames 49 and 50 the uniform (top-left) and AMR (bottom-right) panels dim while
the two SMR panels brighten.)*

**Setup being shown.** The JG05 vortical shwave (`ipert=2`, $64^2$, box $0.5^2$,
$c_s = 1$, $(n_{wx}, n_{wy}) = (-8, 2)$, amp $= 10^{-4}$), so
$k_x(t) = 2\pi(-16 + 6t)$, $k_y = 8\pi$: the wave unwinds through the swing at
$t = 8/3$ and rewinds after. The movie writes one frame per $\Delta t_{\rm frame} = 0.1$
(51 frames, 0-based, so frames 49/50 are $t = 4.9/5.0$); the color scale is **global and
fixed** across frames and panels (a quantile over all frames in
`scripts/make_density_movie_grid.jl`), so the flicker is in the data, not the rendering.

**The relevant physics.** The vortical shwave is the *incompressible* solution; in
compressible hydro its density is a driven oscillator: a slaved compressive response
plus a **free acoustic oscillation** at the instantaneous (WKB) frequency
$\omega(t) = c_s k(t)$. The pgen initializes $\delta\rho = 0$ — not the slaved value —
so the free ringing is launched at $t = 0$ and never decays; $k(t)$ falls to $k_y$ at
the swing (the ringing visibly slows there) and grows linearly afterwards.

**Method.** (a) Recompute $\max|\rho - 1|$ per frame from the four movie runs' own
snapshots; (b) rerun the uniform case with output cadence $0.004$ ($25\times$ finer,
~15 s) and extract the same amplitude — the resolved curve the movie is sampling.


In [ ]:
## Q0.2 — data. (a) Per-frame density amplitudes of the four movie runs
## (validation/movie_runs, 51 frames, output dt = 0.1). (b) A fine-cadence rerun of
## the uniform case (output dt = 0.004, ~15 s in a temp dir; only the extracted
## amplitude series is kept) to resolve what the movie is sampling.
## Caches: run/q02_movie_amps.txt, run/q02_shw_fine_amp.txt.
const MOV = joinpath(VAL, "movie_runs")

"read one scalar + time from an AthenaK vtk (lifted from scripts/make_density_movie_grid.jl)"
function read_vtk_scalar(filename, field)
    data = read(filename)
    header = String(data[1:findfirst(codeunits("CELL_DATA"), data)[1]-1])
    dims = Int[]; t = 0.0
    for line in split(header, "\n")
        startswith(line, "DIMENSIONS") && (dims = parse.(Int, split(line)[2:end]))
        occursin("time=", line) &&
            (t = parse(Float64, match(r"time=\s*([\-0-9.eE+]+)", line).captures[1]))
    end
    nx, ny, nz = max.(dims .- 1, 1)
    idx = findfirst(codeunits("SCALARS $field float"), data)
    nl1 = findnext(==(0x0a), data, idx[end]); nl2 = findnext(==(0x0a), data, nl1 + 1)
    raw = data[nl2+1:nl2+nx*ny*nz*4]
    reshape(Float64.(ntoh.(reinterpret(Float32, raw))), (nx, ny, nz)), t
end

let cf = joinpath(RUN, "q02_movie_amps.txt")
    if !isfile(cf)
        runs = (("shw_uniform", "shwave"), ("shw_smr_ring2", "shwave2_smr_ring2"),
                ("shw_smr_bndry2", "shwave2_smr_bndry2"), ("shw_amr_ring", "shwave2_amr"))
        open(cf, "w") do io
            println(io, "# t  max|dens-1|: uniform  ring2  bndry2  amr")
            for i in 0:50
                @printf(io, "%.2f", 0.1 * i)
                for (d, bn) in runs
                    fb = read_bin_blocks(joinpath(MOV, d, "bin",
                                                  @sprintf("%s.hydro_w.%05d.bin", bn, i)))
                    iv = findfirst(==("dens"), fb.vars)
                    @printf(io, "  %.8e",
                            maximum(maximum(abs.(b.data[:, :, :, iv] .- 1.0)) for b in fb.blocks))
                end
                println(io)
            end
        end
    end
end
let cf = joinpath(RUN, "q02_shw_fine_amp.txt")
    if !isfile(cf)
        ATH = joinpath(REPO, "build", "src", "athena")
        INP = joinpath(REPO, "inputs", "shearing_box", "hydro_incompress_shwave.athinput")
        mktempdir() do td
            run(pipeline(Cmd(Cmd([ATH, "-i", INP, "job/basename=shw_fine",
                                  "output2/dt=0.004"]); dir=td); stdout=devnull))
            open(cf, "w") do io
                println(io, "# t  max|dens-1|   (uniform 64^2 vortical shwave, output dt=0.004)")
                for f in sort(readdir(joinpath(td, "vtk"); join=true))
                    d, t = read_vtk_scalar(f, "dens")
                    @printf(io, "%.6f  %.8e\n", t, maximum(abs.(d .- 1.0)))
                end
            end
        end
    end
end
mv = readcols(joinpath(RUN, "q02_movie_amps.txt"))
println("per-frame density amplitude max|rho-1| (x1e4), movie frames 42-50 (t = 4.2-5.0):")
@printf("%-13s%14s%14s%14s%14s\n", "frame (t)", "uniform", "ring2 SMR", "bndry2 SMR", "AMR ring")
for i in 42:50
    @printf("%3d (t=%.1f) ", i, 0.1 * i)
    for j in 2:5; @printf("%14.4f", 1e4 * mv[i+1, j]); end
    println()
end
println("\nchange from frame 49 to 50 (the PI's frames; movie frames are 0-based):")
for (j, nm) in ((2, "uniform"), (3, "ring2 SMR"), (4, "bndry2 SMR"), (5, "AMR ring"))
    @printf("  %-11s %+0.3e  (%s)\n", nm, 1e4 * (mv[51, j] - mv[50, j]),
            mv[51, j] > mv[50, j] ? "increases" : "decreases")
end

In [ ]:
## Q0.2 — figure + aliasing arithmetic. The density of the vortical-shwave test obeys
## a driven-oscillator equation whose free branch rings at the instantaneous acoustic
## frequency omega(t) = cs*k(t), k(t) = sqrt(kx(t)^2 + ky^2), kx(t) = kx0 + q*Om*ky*t.
## The movie samples it at dt_frame = 0.1.
csnd, qsh, Om = 1.0, 1.5, 1.0
kx0, ky = 2π * (-8) / 0.5, 2π * 2 / 0.5
kx(t) = kx0 + qsh * Om * ky * t
kk(t) = sqrt(kx(t)^2 + ky^2)

fine = readcols(joinpath(RUN, "q02_shw_fine_amp.txt"))
mv   = readcols(joinpath(RUN, "q02_movie_amps.txt"))   # t, unif, ring2, bndry2, amr

# measured oscillation period near t = 4.9 from successive minima of the fine series
s = (fine[:, 1] .> 4.55) .&& (fine[:, 1] .< 5.0)
tf, af = fine[s, 1], fine[s, 2]
imins = [i for i in 2:length(af)-1 if af[i] < af[i-1] && af[i] < af[i+1]]
Tmeas = length(imins) > 1 ? (tf[imins[end]] - tf[imins[1]]) / (length(imins) - 1) : NaN
Twkb = π / (csnd * kk(4.775))    # |rho-1| beats at TWICE the acoustic frequency
@printf("acoustic frequency: cs*k = %.1f (t=4.55) rising to %.1f (t=5.0) — a chirp\n",
        csnd * kk(4.55), csnd * kk(5.0))
@printf("period of the observable |rho-1|: WKB pi/(cs k) = %.4f at mid-window;\n", Twkb)
@printf("measured minima spacing of the fine series (t=4.55-5.0): %.4f\n", Tmeas)
@printf("frame cadence 0.1 = %.1f-%.1f beat periods per frame (Nyquist needs < 0.5):\n",
        0.1 / (π / (csnd * kk(4.55))), 0.1 / (π / (csnd * kk(5.0))))
println("consecutive frames sample a chirping oscillation at quasi-random phases —")
println("the flicker, its apparent 2-3-frame rhythm, and its run-to-run sign")
println("disagreements are all stroboscopic artifacts of the output cadence.")

fig = Figure(size=(980, 620))
ax1 = Axis(fig[1, 1]; xlabel="t", ylabel="max |ρ − 1|  × 10⁴",
           title="the movie's frame cadence (points) vs the resolved density oscillation (line)")
lines!(ax1, fine[:, 1], 1e4 .* fine[:, 2]; color=(:gray35, 0.9), linewidth=1,
       label="uniform, output dt = 0.004")
for (j, nm, c, mk) in ((2, "uniform", C1, :circle), (3, "ring2 SMR", C3, :rect),
                       (4, "bndry2 SMR", C4, :diamond), (5, "AMR ring", C2, :utriangle))
    scatterlines!(ax1, mv[:, 1], 1e4 .* mv[:, j]; color=c, marker=mk, markersize=9,
                  linewidth=0.8, linestyle=:dot, label=nm)
end
xlims!(ax1, 4.25, 5.02)
axislegend(ax1; position=:lt, framevisible=false, labelsize=10, nbanks=2)

ax2 = Axis(fig[2, 1]; xlabel="t", ylabel="max |ρ − 1|  × 10⁴", height=170,
           title="full run: the swing (t = 2.67) launches the free acoustic ringing; k(t) keeps rising")
lines!(ax2, fine[:, 1], 1e4 .* fine[:, 2]; color=(:gray35, 0.9), linewidth=0.7)
scatter!(ax2, mv[:, 1], 1e4 .* mv[:, 2]; color=C1, markersize=5)
vlines!(ax2, [8π / (3π)]; color=(:gray, 0.4), linestyle=:dash)  # t_swing = 32π/12π
fig

**Answer: the flicker is temporal aliasing of a real, smooth, bounded acoustic
oscillation — not a refinement artifact and not a rendering artifact.**

- The per-frame table reproduces the PI's observation exactly (frames 49→50: uniform
  $-1.9\times10^{-5}$ and AMR $-1.1\times10^{-5}$ *down*; ring2 $+0.8\times10^{-5}$ and
  bndry2 $+1.0\times10^{-5}$ *up*) — and shows the same erratic hopping at *every*
  frame pair, in the **uniform run most of all**. Whatever it is, it is not the
  refinement machinery.
- The fine-cadence rerun resolves what the movie samples: $\max|\rho-1|$ oscillates
  with a smooth envelope ($\sim(0.05$–$0.3)\times10^{-4}$ after the swing) at the WKB
  acoustic frequency — $c_s k = 75\to92$ over $t = 4.55\to5.0$, i.e. **2.4–2.9
  oscillations of the observable per movie frame** (Nyquist needs < 0.5). Consecutive
  frames therefore sample a *chirping* oscillation at quasi-random phases: stroboscopic
  flicker, with no clean alias period. The uniform run's movie points lie exactly on
  the fine curve (deterministic rerun; frame-50 values agree to all printed digits).
- **Why the panels disagree in direction:** each run rings at a slightly different
  accumulated acoustic phase, because the phase error scales with the timestep
  (exactly the RK2 $\propto\Delta t^2$ drift quantified in the Q0.1 extension). The
  two 2-level runs share the *same* global $\Delta t$ (set by their finest level), so
  they are phase-locked to each other — their amplitudes agree to a few percent at
  every frame and they flicker in the same direction; uniform ($\Delta t = 2.3\times
  10^{-3}$) and the 1-level AMR run ($\Delta t/2$) happen to sit near each other in
  phase at these frames. Under $2$–$3\times$ undersampling, any phase offset of a
  fraction of a period flips the apparent direction.
- Nothing physical flickers: the resolved envelope is smooth, bounded, and identical
  in character across the four meshes.

**Practical note for movies of this test:** resolve the ringing (output
$\mathrm{d}t \lesssim 0.01$ at these $k$) or render the envelope; at $\mathrm{d}t=0.1$
the panel brightness is a stroboscope, not a diagnostic.


#### Background — why the density peaks when it does, and why it decays

*(study note: the envelope physics of the vortical-shwave density, tested
parameter-free against the fine-cadence series below)*

**1. The system.** Linear isothermal perturbations on the shear, as a single shearing
wave $\propto \exp[i(k_x(t)x + k_y y)]$ with $k_x(t) = k_{x0} + q\Omega k_y t$
(the shear tilts the wave; here $k_x/k_y$ runs from $-4$ through $0$ at
$t_{\rm swing}=8/3$ and back up). The Fourier amplitudes obey
$$
\dot{\hat v}_x = 2\Omega \hat v_y - i k_x c_s^2 \hat\delta,\qquad
\dot{\hat v}_y = -(2{-}q)\Omega \hat v_x - i k_y c_s^2 \hat\delta,\qquad
\dot{\hat\delta} = -i(k_x \hat v_x + k_y \hat v_y),
$$
with $\hat\delta = \delta\rho/\rho_0$.

**2. The carrier: potential vorticity.** The combination
$\xi = i(k_x\hat v_y - k_y \hat v_x) - (2{-}q)\Omega\,\hat\delta$ (perturbation PV) is
**exactly conserved**. In the nearly incompressible regime the vortical wave has
$\hat v_x = ik_y\hat\psi$, $\hat v_y = -ik_x\hat\psi$ with $\xi \simeq k^2\hat\psi$, so
$$
|\mathbf{v}| = \frac{\xi}{k(t)}:
$$
the velocity amplitude **peaks where $k(t)$ is smallest** — at the swing, $k_x = 0$,
$k = k_y$ — and decays as $1/k \propto 1/t$ afterwards as the shear rewinds the wave to
ever finer radial scales. Transient amplification, no instability: the PV is fixed and
gets spread over growing $k$.

**3. The density: a driven oscillator.** Differentiating $\hat\delta$'s equation once,
$$
\ddot{\hat\delta} + c_s^2 k(t)^2\, \hat\delta = S(t), \qquad
S = \frac{2\Omega\,\xi\,\big[(q{-}1)k_y^2 - k_x(t)^2\big]}{k(t)^2},
$$
i.e. an acoustic oscillator at the instantaneous frequency $c_s k(t)$, driven by the
Coriolis/shear forces acting on the vortical flow. Its solution =
**slaved response + free ringing**:
- *slaved* (adiabatic, valid since $c_s k \gg \Omega$ throughout):
$$
\hat\delta_{\rm slv}(t) = \frac{S}{c_s^2k^2}
= \frac{2\Omega\,\xi\,\big[(q{-}1)k_y^2 - k_x^2\big]}{c_s^2\,k^4},
\qquad \xi = \mathrm{amp}\cdot \frac{k_0^2}{k_y};
$$
- *free ringing* at $\omega = c_s k(t)$, launched at $t=0$ because the pgen initializes
  $\delta\rho = 0$ instead of $\hat\delta_{\rm slv}(0)$ (and re-pumped near the swing,
  where the drive varies fastest); WKB amplitude $\propto k^{-1/2}$.

**4. The envelope anatomy** (all parameter-free; overlay in the next cell):
- rising branch and a **local max at $k_x = -\sqrt2\,k_y$** ($t = 1.72$): maximize
  $|k_x^2 - (q{-}1)k_y^2|/k^4$;
- **nulls at $k_x^2 = (q{-}1)k_y^2$** ($t = 2.20$ and $3.14$): the source changes sign
  — the tilt angle at which the Coriolis force on $\hat v_x$ and the shear/Coriolis
  force on $\hat v_y$ cancel;
- the **absolute peak at the swing** ($t = 8/3$), where $1/k^4$ is maximal:
  $\hat\delta_{\rm peak} = 2\Omega\xi(q{-}1)/(c_s^2k_y^2) = 0.68\times10^{-4}$ here
  (measured $0.76\times10^{-4}$, the excess being the ringing riding on top). The
  density peaks at the swing for the same reason the velocity does — minimum $k$ —
  but more sharply ($k^{-4}$ vs $k^{-1}$), because pressure resists compression less
  when the wave is least wound;
- **decay as $t\to\infty$**: the slaved part falls as
  $2\Omega\xi/(c_s k_x)^2 \propto t^{-2}$ (pressure stiffens as $c_s^2k^2$ while the
  vortical drive weakens), the ringing as $k^{-1/2} \propto t^{-1/2}$ — so the late
  band is ringing-dominated. On longer horizons the winding drives the radial
  wavelength to the grid scale and numerical diffusion ($\propto k^2$) extinguishes
  everything super-exponentially. Nothing can grow: without self-gravity there is no
  feedback channel (contrast the `swing_selfgrav` runs at $4\pi G > 2$, where the
  crest grows instead of bouncing).

**References.** The derivation is self-contained, but the framework is the "shwave"
linear theory of Johnson & Gammie (2005a, ApJ 626, 978 — cited by `shwave.cpp`); the
test problem is the code-verification setup of Johnson & Gammie (2005b, ApJ 635, 149,
their Fig. 1) — the PV result in component form, $\hat v_x \propto (1+\tau^2)^{-1}$,
is exactly their eq. (9), verified to <0.5% in the validation document. See also
Goldreich & Lynden-Bell (1965, MNRAS 130, 125) for swing amplification and Bodo et al.
(2005, A&A 437, 9) for vortex-driven density-wave emission. Both JG05 PDFs are in
`docs/`.


In [ ]:
## Background (Q0.2 continued) — what sets the density envelope: the slaved response
## of the winding vortical wave. Derivation in the markdown cell above; summary:
##   PV conservation: xi = zeta_hat = k(t)^2 psi_hat = const  (incompressible limit)
##   driven oscillator: d2(delta)/dt2 + cs^2 k^2 delta = S,
##       S = 2 Omega xi [ (q-1) ky^2 - kx(t)^2 ] / k(t)^2
##   adiabatic (slaved) solution: delta_slaved = S / (cs^2 k^2)
## with xi = amp * k0^2 / ky (the pgen's amp = 1e-4 is the vx amplitude).
## Everything below is PARAMETER-FREE — no fitting.
amp_shw = 1.0e-4
k0 = kk(0.0)
xi = amp_shw * k0^2 / ky
dslv(t) = 2 * Om * xi * abs((qsh - 1) * ky^2 - kx(t)^2) / (csnd^2 * kk(t)^4)

# predicted special times: kx/ky = -sqrt(2) (envelope local max), +-sqrt(q-1) (nulls),
# 0 (swing peak); kx(t)/ky = kx0/ky + q*Om*t
tof(v) = (v - kx0 / ky) / (qsh * Om)
t_bump, t_null1, t_swing, t_null2 = tof(-√2), tof(-√(qsh - 1)), tof(0.0), tof(√(qsh - 1))
@printf("predicted: local max t=%.3f, nulls t=%.3f / %.3f, swing peak t=%.3f\n",
        t_bump, t_null1, t_null2, t_swing)
@printf("predicted amplitudes: at t=0 %.3f, bump %.3f, swing peak %.3f  (x1e-4)\n",
        1e4 * dslv(0), 1e4 * dslv(t_bump), 1e4 * dslv(t_swing))
i_pk = argmax(fine[:, 2])
@printf("measured : swing peak %.3f at t=%.3f (slaved %.3f + free ringing on top)\n",
        1e4 * fine[i_pk, 2], fine[i_pk, 1], 1e4 * dslv(t_swing))
@printf("late-time slaved tail: dslv ~ 2*Om*xi/(cs*kx)^2 ~ t^-2: %.4f (t=4) -> %.4f (t=5) x1e-4\n",
        1e4 * dslv(4.0), 1e4 * dslv(5.0))

fig = Figure(size=(980, 420))
ax = Axis(fig[1, 1]; xlabel="t", ylabel="max |ρ − 1|  × 10⁴",
          title="measured density amplitude vs the parameter-free slaved envelope (no fit)")
lines!(ax, fine[:, 1], 1e4 .* fine[:, 2]; color=(:gray55, 0.8), linewidth=0.7,
       label="measured (output dt = 0.004)")
tt = range(0, 5; length=2000)
lines!(ax, tt, 1e4 .* dslv.(tt); color=C2, linewidth=2.5,
       label="slaved envelope  2Ωξ|(q−1)k_y² − k_x²| / c_s²k⁴")
vlines!(ax, [t_bump, t_null1, t_swing, t_null2]; color=(:gray, 0.5), linestyle=:dash)
for (tv, lb) in ((t_bump, "local max"), (t_null1, "null"), (t_swing, "swing"), (t_null2, "null"))
    text!(ax, tv + 0.03, 0.68; text=lb, color=:gray30, fontsize=10)
end
axislegend(ax; position=:lt, framevisible=false)
fig

## Phase 1 — shear-periodic multigrid boundary conditions

*(questions to be added)*


## Phase 2 — multigrid gravity on refined shearing meshes

*(questions to be added)*


### Q2.1 — Are the y-boundary collapse clumps a code artifact?

*(as posed 2026-08-19: in every collapse run — uniform $256^2$ MG, AMR $128^2$, FFT
$256^2$ — the dominant clump forms straddling $y = \pm\pi$; the PI wants to rule out
anything wrong in the code.)*

**Verified positions** (column-max argmax from the cached bins):

| run | dominant clump (t = 10) | site |
|---|---|---|
| uniform $128^2$ | 131 $\rho_0$ at $(+0.025, -0.025)$ | center, $y \approx 0$ |
| AMR $128^2$+L1 | 152 $\rho_0$ at $(-0.012, -3.105)$ | wrap, $y \approx -\pi$ |
| MG $256^2$ (NAS) | 40.4 $\rho_0$ at $(-0.012, -2.884)$ + mirror | straddling $y=\pm\pi$ |
| FFT $256^2$ | 41.9 $\rho_0$ at $(-0.012, -2.859)$ + mirror | straddling $y=\pm\pi$ |

**Evidence chain below**: (a) the swing flips the wave's sign, so the amplified crest
at the corotation column sits at $y = \pm\pi$; (b) the exact mirror symmetry
quantizes the corotation crest to the two fixed points $\{0, \pi\}$ — the only
candidate collapse sites, both realized across the runs; (c) visualization;
(d) the decisive translation-covariance test with a phase-shifted initial wave.


In [ ]:
## Q2.1(a) — the swing SIGN FLIP. The pgen's IC puts the wave crest of the
## corotation column (x = 0, where phase = ky*y) at y = 0. But d_cos < 0 through
## the entire swing + fragmentation window: the amplified wave is ANTI-phased with
## the IC, so at fragmentation the crest sits at cos(y) = -1, i.e. y = +-pi.
for (nm, f) in (("uniform 128^2", "clump_unif.user.hst"),
                ("uniform 256^2 (NAS)",
                 joinpath("swing_mg_uniform_256_tlim14", "SwingSG.user.hst")))
    h = read_hst(joinpath(RUN, f))
    @printf("%-20s", nm)
    for tt in (3.0, 4.0, 4.5, 5.0, 6.0)
        i = findfirst(>=(tt), h[:, 1])
        @printf("  d_cos/A(t=%.1f) = %+7.3f", tt, h[i, 3] / 0.2)
    end
    println()
end
println("(fragmentation happens at t ≈ 4.5, deep in the d_cos < 0 phase)")

In [ ]:
## Q2.1 prep — harmonic PHASES of the 256^2 collapse: where is the m=1 crest in y,
## and is the m=2 fragmentation harmonic phase-locked to it (align = phi2 - 2*phi1)?
## Caches: run/q21_phases_{mg,fft}.txt.
using Statistics
wrap(a) = mod(a + π, 2π) - π
for (dir, tag) in ((D256MG, "mg"), (D256FFT, "fft"))
    cf = joinpath(RUN, "q21_phases_$tag.txt")
    isfile(cf) && continue
    fs = sort(filter(f -> startswith(f, "SwingSG.hydro_u_d"), readdir(joinpath(dir, "bin"))))
    open(cf, "w") do io
        println(io, "# t  phi1  phi2  y_crest(m1, x=0 col)  align(phi2-2phi1)  |c1|  |c2|")
        for f in fs
            t = 0.05 * parse(Int, split(f, '.')[end-1])
            A = mosaic(joinpath(dir, "bin", f), "dens")
            S = dropdims(mean(A; dims=3); dims=3)
            nx, ny = size(S)
            # y_j = -π + (j-1/2)*2π/ny; use e^{-i m y_j} so that S ~ cos(m(y-y0))
            # gives c_m ∝ e^{-i m y0} and y0 = -arg(c_m)/m directly in box coords
            yj = [-π + 2π * (j - 0.5) / ny for j in 1:ny]
            # anchor at the corotation column x ~ 0 (mean of the two center columns,
            # where the background shear vanishes and the knots live)
            row = vec(mean(S[nx÷2:nx÷2+1, :]; dims=1))
            c1 = sum(row .* cis.(-1 .* yj)); c2 = sum(row .* cis.(-2 .* yj))
            f1, f2 = angle(c1), angle(c2)
            ix = nx ÷ 2
            println(io, join([@sprintf("%.4f", v) for v in
                              (t, f1, f2, wrap(-f1), wrap(f2 - 2*f1),
                               abs(c1) / ny, abs(c2) / ny)], "  "))
        end
    end
    println("wrote $cf")
end

ph = readcols(joinpath(RUN, "q21_phases_mg.txt"))
sel = (ph[:, 1] .>= 2.5) .&& (ph[:, 1] .<= 6.0)
println("swing+fragmentation window t = 2.5–6.0 ($(count(sel)) snapshots), corotation column:")
println("  distinct y_crest values: ", sort(unique(round.(ph[sel, 4]; digits=3))),
        "   (m=1 crest pinned AT the wrap)")
s2 = sel .&& (ph[:, 7] .> 0.02)
println("  distinct |phi2| values (where |c2| > 0.02): ",
        sort(unique(round.(abs.(ph[s2, 3]); digits=3))), "   (m=2 maxima at 0 AND ±π)")
@printf("  |c1| grows %.3f → %.3f across the window\n",
        ph[findfirst(sel), 6], maximum(ph[sel, 6]))

**Why the phases are quantized (the parity argument).** The IC
$\rho = 1 + A\cos(k_{x0}x + k_y y)$ is even under the point reflection
$P:(x,y)\to(-x,-y)$; the shearing-box equations commute with $P$; and the code
preserves the symmetry **bitwise** (checked through $t = 12.5$, deep past
fragmentation — itself a strong code check: any site-dependent asymmetry in the
$y$-wrap, FARGO shift, or ghost exchange would break it). The corotation column maps
to itself with $y \to -y$, so its density profile is an even function of $y$; an even
function's Fourier coefficients are real, so their phases can only be $0$ or $\pi$:
the $m{=}1$ crest can sit **at** $y=0$ ($c_1>0$) or **at** $y=\pi$ ($c_1<0$) and
nowhere else — these are the two fixed points of the reflection. Structure anywhere
else must come as mirror pairs (as the off-column fragments do). During the swing the
sign flip makes $c_1 < 0$ (crest at $\pi$) while $c_2 > 0$ ($m{=}2$ maxima at $0$
*and* $\pi$): exactly two candidate collapse sites, and every run collapsed onto one
of them — the resolved runs at $\pi$ (crest + harmonic), the under-resolved $128^2$
uniform at $0$ (harmonic only).

In [ ]:
## Q2.1 — visualization of the phase-quantization finding.
using Statistics
wrap(a) = mod(a + π, 2π) - π

# (A) corotation-column density profiles at several times (256^2 MG)
prof(idx) = begin
    A = mosaic(joinpath(D256MG, "bin", @sprintf("SwingSG.hydro_u_d.%05d.bin", idx)), "dens")
    S = dropdims(mean(A; dims=3); dims=3)
    vec(mean(S[128:129, :]; dims=1))
end
ys = [-π + 2π * (j - 0.5) / 256 for j in 1:256]

# (B) phase history
ph = readcols(joinpath(RUN, "q21_phases_mg.txt"))

# (C) the t=4.5 field with the parity structure
A45 = mosaic(joinpath(D256MG, "bin", "SwingSG.hydro_u_d.00090.bin"), "dens")
S45 = dropdims(mean(A45; dims=3); dims=3)

fig = Figure(size=(1080, 800))

axA = Axis(fig[1, 1:2]; xlabel="y", ylabel="⟨ρ⟩_z at x ≈ 0",
           title="(A) corotation-column density: the amplified crest grows AT y = ±π",
           xticks=([-π, -π/2, 0, π/2, π], ["−π", "−π/2", "0", "π/2", "π"]))
for (idx, c, lw) in ((60, (:gray, 0.6), 1.2), (80, C3, 1.5), (90, C4, 1.8), (100, C2, 2.2))
    lines!(axA, ys, prof(idx); color=c, linewidth=lw, label=@sprintf("t = %.1f", 0.05 * idx))
end
vlines!(axA, [-π + π/256, π - π/256]; color=(:black, 0.4), linestyle=:dash)
vlines!(axA, [0]; color=(:black, 0.25), linestyle=:dot)
text!(axA, -3.05, 7.2; text="wrap (fixed point)", fontsize=10, color=:gray30)
text!(axA, 0.05, 7.2; text="center (fixed point)", fontsize=10, color=:gray30)
axislegend(axA; position=:lt, framevisible=false)

axB = Axis(fig[2, 1:2]; xlabel="t", ylabel="crest position  −φ₁ (wrapped)",
           title="(B) the m=1 crest position is QUANTIZED: exactly 0 or ±π, nothing between",
           yticks=([-π, 0, π], ["−π", "0", "π"]))
c1n = ph[:, 6] ./ maximum(ph[:, 6])
scatter!(axB, ph[:, 1], ph[:, 4]; color=[(C1, clamp(0.15 + 3v, 0.15, 1.0)) for v in c1n],
         markersize=7)
band!(axB, [2.5, 6.0], [-3.6, -3.6], [3.6, 3.6]; color=(C2, 0.08))
text!(axB, 2.6, 2.2; text="swing + fragmentation\n(marker opacity ∝ |c₁|)", fontsize=10,
      color=:gray30)
ylims!(axB, -3.7, 3.7)

axC = Axis(fig[3, 1]; xlabel="x", ylabel="y", aspect=DataAspect(),
           title="(C) t = 4.5: field is exactly mirror-symmetric about (0,0)")
hm = heatmap!(axC, range(-π, π; length=256), range(-π, π; length=256), S45;
              colormap=:viridis)
scatter!(axC, [0.0, 0.0, 0.0], [0.0, π - 0.02, -π + 0.02]; color=:red, marker=:star5,
         markersize=14)
arrows2d!(axC, [1.4], [1.9], [-2.8], [-3.8]; color=(:red, 0.7))
scatter!(axC, [1.4, -1.4], [1.9, -1.9]; color=:white, strokecolor=:red, strokewidth=1.5,
         markersize=9)
Colorbar(fig[3, 0], hm; label="⟨ρ⟩_z")

axD = Axis(fig[3, 2]; xlabel="y", ylabel="mode amplitude at x ≈ 0", yscale=log10,
           title="(D) why two candidate sites: m=1 picks π, m=2 supports 0 AND π",
           xticks=([-π, 0, π], ["−π", "0", "π"]))
row45 = vec(mean(S45[128:129, :]; dims=1))
c1v = sum(row45 .* cis.(-1 .* ys)) / 256
c2v = sum(row45 .* cis.(-2 .* ys)) / 256
yy = range(-π, π; length=400)
lines!(axD, yy, 1 .+ 2 * real(c1v) .* cos.(yy) .+ 2 * real(c2v) .* cos.(2 .* yy);
       color=:black, linewidth=2, label="1 + m=1 + m=2 reconstruction")
lines!(axD, yy, max.(2 * abs(real(c1v)) .* cos.(yy .- π) .+ 1, 1e-2);
       color=(C1, 0.6), linestyle=:dash, label="m=1 alone (crest at ±π)")
lines!(axD, yy, max.(2 * abs(real(c2v)) .* cos.(2 .* yy) .+ 1, 1e-2);
       color=(C2, 0.6), linestyle=:dot, label="m=2 alone (crests at 0 and ±π)")
axislegend(axD; position=:cb, framevisible=false, labelsize=9)
fig

**The decisive test: translation covariance.** All the above is
consistency evidence; the falsification test is to *move the pattern* and see whether
the clumps follow it or stay glued to the wrap. The swing pgen gained a
`<problem> phase0` parameter (2026-08-19, uncommitted): the IC becomes
$\cos(k_{x0}x + k_y y + \varphi_0)$ and the history projection carries the same
shift, so the $\varphi_0$ solution must be the $\varphi_0{=}0$ solution rigidly
translated by $\Delta y = -\varphi_0/k_y$ (both $\pi$ and $\pi/2$ are integer-cell
shifts at $128^2$). Predictions: $\varphi_0 = \pi$ moves the fragmenting crest to the
*center*-adjacent site; $\varphi_0 = \pi/2$ moves the whole pattern — fixed points
included — to $y = \mp\pi/2$, away from both center and boundary. If a clump stuck to
$y = \pm\pi$ regardless, the wrap would be guilty.

In [ ]:
## Q2.1(d) — the decisive translation-covariance test. The swing pgen gained a
## <problem> phase0 parameter (IC cos(kx0 x + ky y + phase0), same shift in the
## history projection; default 0.0 is bitwise-inert). If the wrap had any pull,
## clumps would stick to y = +-pi; if the physics owns the sites, the whole
## pattern must translate rigidly by -phase0/ky. Runs (~8 min each, uniform 128^2):
##   athena -i inputs/tests/swing_selfgrav_clump_amr.athinput job/basename=clump_ph_pi \
##     mesh_refinement/refinement=none problem/phase0=3.141592653589793 \
##     output2/dt=0.25 output3/dt=100.0        # and phase0=pi/2 -> clump_ph_pi2
co(i, n) = -π + (i - 0.5) * 2π / n
wrap(a) = mod(a + π, 2π) - π
colmax128(bn, idx) = begin
    A = mosaic(joinpath(RUN, "bin", @sprintf("%s.hydro_u_d.%05d.bin", bn, idx)), "dens")
    dropdims(maximum(A; dims=3); dims=3)
end
Aur = colmax128("clump_unif", 4)                 # phase0 = 0 reference, t = 10
ir = argmax(Aur)
yref = co(ir[2], 128)
@printf("reference (phase0=0):  max rho %.2f at (%+.3f, %+.3f)\n",
        Aur[ir], co(ir[1], 128), yref)
for (bn, ph0) in (("clump_ph_pi", π), ("clump_ph_pi2", π / 2))
    M = colmax128(bn, 40)
    i = argmax(M)
    @printf("%-13s (t=10): max rho %.2f at (%+.3f, %+.3f)   predicted y = %+.3f\n",
            bn, M[i], co(i[1], 128), co(i[2], 128), wrap(yref - ph0))
end
println("(positions match the rigid shift to the CELL, modulo each run's own mirror")
println(" degeneracy — the phase0=pi pattern is still parity-symmetric, so roundoff")
println(" picks which of the two mirror twins wins the runaway)")

# the histories must be the SAME computation, translated: the projection carries
# phase0, so d_cos must reproduce the reference run's numbers
hu = read_hst(joinpath(RUN, "clump_unif.user.hst"))
for bn in ("clump_ph_pi", "clump_ph_pi2")
    h = read_hst(joinpath(RUN, bn * ".user.hst"))
    nr = min(size(h, 1), size(hu, 1))
    s6 = hu[1:nr, 1] .<= 6.0
    @printf("%-13s max|d_cos - ref|/A:  t <= 6: %.1e   full run: %.1e\n", bn,
            maximum(abs.(h[1:nr, 3][s6] .- hu[1:nr, 3][s6])) / 0.2,
            maximum(abs.(h[1:nr, 3] .- hu[1:nr, 3])) / 0.2)
end
println("(bitwise-identical through fragmentation; the late 1e-8-level gap is roundoff")
println(" amplified by the runaway — same max rho to all printed digits)")

**Verdict — the $y$-wrap is fully exonerated.**
- $\varphi_0 = \pi/2$ (the parity-breaking case, no mirror ambiguity): clump at
  $(+0.025, -1.595)$ vs predicted $-1.595$ — **the exact predicted cell**, at
  $y = -\pi/2$, far from center and boundary alike.
- $\varphi_0 = \pi$: clump at $(-0.025, -3.117)$ vs predicted $(+0.025, +3.117)$ —
  the parity **mirror twin** of the prediction, one cell across the wrap (the
  $\varphi_0{=}\pi$ pattern is still parity-symmetric, so the collapse forms a mirror
  dumbbell and round-off decides which twin wins the runaway — the same coin-flip
  seen between $y=0$ and $y=\pi$ across the original runs).
- The histories are the **same computation, translated**: $d_{\cos}$ bitwise
  identical to the $\varphi_0{=}0$ reference through $t \le 6$ (through
  fragmentation), $5\times10^{-8}$/$5\times10^{-10}$ over the full run (round-off
  amplified by the runaway), and max $\rho$ identical to all printed digits (131.08)
  in all three runs.

The clumps sit where the amplified wave's symmetry puts them: the swing sign flip
parks the crest at the parity fixed point that happens to coincide with the wrap
under the pgen's $\cos$ convention. Move the wave's phase and the clumps move with
it, cell-exactly. Combined with the bitwise parity conservation, the MG-vs-FFT
$1/3$-cell agreement (two entirely different $y$-implementations), and the serial
vs 64-rank agreement, there is **no code preference for the boundary**.
*(Uncommitted artifacts: `swing.cpp` + input `phase0`; runs `clump_ph_pi{,2}` and
caches `q21_phases_{mg,fft}.txt` in `validation/run`.)*

#### Follow-up — three questions on the mechanism (2026-08-19)

*(1) Both $128^2$ and $256^2$ started at zero phase — why does one collapse at $y=0$
and the other at $y=\pm\pi$? (2) Why always the corotation column? (3) The collapse is
highly nonlinear — why do the initial perturbations still dictate the clump sites?*


In [ ]:
## Q2.1 follow-up (i) — the site-selection mechanism: post-bounce slosh vs runaway
## onset. The 128^2 corotation phases come from the clump_ph_0 frames (cache
## run/q21_slosh128.txt); 256^2 from the q21_phases cache. Onset = last upward
## crossing of max rho = 20.
function slosh128()
    cf = joinpath(RUN, "q21_slosh128.txt")
    if !isfile(cf)
        open(cf, "w") do io
            println(io, "# t  y_crest  |c1|   (corotation column of clump_ph_0)")
            for idx in 0:40
                A = mosaic(joinpath(RUN, "bin",
                                    @sprintf("clump_ph_0.hydro_u_d.%05d.bin", idx)), "dens")
                S = dropdims(mean(A; dims=3); dims=3)
                yj = [-π + 2π * (j - 0.5) / 128 for j in 1:128]
                row = vec(mean(S[64:65, :]; dims=1))
                c1 = sum(row .* cis.(-1 .* yj)) / 128
                @printf(io, "%.4f  %.4f  %.6e\n", 0.25 * idx, wrap(-angle(c1)), abs(c1))
            end
        end
    end
    readcols(cf)
end
sl = slosh128()
ph_mg = readcols(joinpath(RUN, "q21_phases_mg.txt"))
r128s = readcols(joinpath(RUN, "clump_unif.rhomax.txt"))
rmg_s = readcols(joinpath(RUN, "swing_mg_uniform_256_tlim14", "rhomax.txt"))
onset(t, r) = t[findlast(i -> r[i] < 20 && r[i+1] >= 20, 1:length(r)-1)]
t_on128, t_on256 = onset(r128s[:, 1], r128s[:, 3]), onset(rmg_s[:, 1], rmg_s[:, 2])
@printf("runaway onset (max rho crosses 20): 128^2 t = %.2f    256^2 t = %.2f\n",
        t_on128, t_on256)

fig = Figure(size=(1000, 640))
for (p, ttl, tt, aa, yc, rt, rr, tons) in
    ((1, "uniform 128²  —  onset t = 8.8 catches the slosh at y = 0 → clump at CENTER",
      sl[:, 1], sl[:, 3], sl[:, 2], r128s[:, 1], r128s[:, 3], t_on128),
     (2, "uniform 256²  —  onset t = 9.45, half a slosh later, settles at y = π → knot at WRAP",
      ph_mg[:, 1], ph_mg[:, 6], ph_mg[:, 4], rmg_s[:, 1], rmg_s[:, 2], t_on256))
    ax = Axis(fig[p, 1]; xlabel=p == 2 ? "t" : "", ylabel="|c₁| at x ≈ 0", yscale=log10,
              title=ttl, titlesize=13)
    cols = [abs(v) > 1.5 ? C2 : C1 for v in yc]
    scatter!(ax, tt, max.(aa, 1e-3); color=cols, markersize=6)
    lines!(ax, tt, max.(aa, 1e-3); color=(:gray, 0.3), linewidth=0.7)
    axr = Axis(fig[p, 1]; yaxisposition=:right, yscale=log10, ylabel="max ρ/ρ₀",
               ylabelcolor=:gray40, yticklabelcolor=:gray40)
    hidespines!(axr); hidexdecorations!(axr)
    lines!(axr, rt, rr; color=(:gray40, 0.8), linewidth=1.6)
    vlines!(ax, [tons]; color=(:red, 0.6), linestyle=:dash, linewidth=1.5)
    xlims!(ax, 0, 11); xlims!(axr, 0, 11); ylims!(ax, 1e-3, 20)
    p == 1 && text!(ax, 0.3, 6;
                    text="blue: crest at y = 0    orange: crest at y = ±π    gray: max ρ",
                    fontsize=11)
end
fig

**(1) The site is chosen during the post-bounce slosh, not at the swing.**
Both runs over-compress at the swing and *bounce* (max $\rho$ back to $\sim$2 by
$t=6$–8). The debris then sloshes: the corotation $m{=}1$ content oscillates between
the two fixed points with period $\sim$1.5 (the rewound wave has $k \approx 4.6$, so
$\omega^2 = \kappa^2 + c_s^2k^2 - 4\pi G \approx +16$ — a *stable* gravito-acoustic
standing oscillation). The runaway fires when a local knot finally goes
Jeans-unstable, and it locks onto whichever site the slosh favors at that moment:
$128^2$ fires at $t = 8.80$ on a $y{=}0$ phase (center clump); $256^2$ fires at
$t = 9.45$, half a slosh cycle later, settling at $\pi$ (wrap knot). The two
resolutions decouple *after* the bounce — the bounce compresses to near the grid
scale at $128^2$ (the documented Truelove marginality) — so the 0-vs-$\pi$ decision,
a symmetric double well, is resolution-sensitive. The converged answer is $\pi$: MG
$256^2$, FFT $256^2$, and the AMR run (fine level $\equiv 256^2$) all agree.

**(2) Why the corotation column.** Rigorously: both parity fixed points lie *on*
$x=0$ (the IC's symmetry center is the origin), and while parity holds a single,
unpaired knot **must** sit at a fixed point — anything off-column can only exist as
an equal-mass mirror pair (the transient census pairs at $\pm(0.01$–$0.09, y)$).
Dynamically: corotation material has zero background drift; a fragment at $|x|>0$
slides azimuthally at $q\Omega|x|$ while its mirror twin slides the other way, so
only the on-column site enjoys stationary, symmetric accretion from both crest arms.
Empirically: every local maximum in the $t=10$ census sits within $|x| \le 0.09$, in
both solvers.

**(3) Nonlinearity scrambles amplitudes and timings, not symmetries.** Translation
covariance and parity commute with the *full* equations at any amplitude — measured:
parity at the float32 floor through $\rho$-contrast $10^{2.5}$, and the
$\varphi_0$-translated runs cell-exact at $\rho = 131$. What nonlinearity is free to
scramble, it demonstrably does: which mirror twin wins (round-off coin flip), the
peak values (MG 40.4 vs FFT 41.9), the past-ceiling density ($2.6\times10^5$ vs
$\sim$700), and — per (1) — which of the two allowed sites is selected. The clump
*location* is a discrete, symmetry-protected label, immune to chaos until round-off
breaks the symmetry itself. And there is no competition: the IC is one coherent mode
at amplitude 0.2 over round-off noise at $10^{-16}$; even at the fastest growth rate
($\sim$2 per time unit) for the whole run, noise reaches only $\sim 10^{-7}$ while
the seed's harmonics start at $10^{-2}$. In a turbulent disk with broadband noise
the sites *would* be stochastic; here "where" is inherited through symmetry, "when
and how deep" is not.

**How the parity conservation is measured.** The reflection is
$P:(x,y)\to(-x,-y)$. On the cell-centered grid no cell sits at the origin — centers
are at $\pm\Delta/2, \pm3\Delta/2,\ldots$ — so cell $i$ maps exactly onto cell
$\mathrm{mod1}(257-i,\,256)$ (an off-by-one here masquerades as a large false
asymmetry). The per-snapshot metric is
$a(t) = \max_{i,j}\,|S(i,j) - S(P(i,j))|$ on the $z$-averaged density: $a=0$ means
the two mirror halves of the stored field are bitwise identical. Because snapshots
are float32, $a$ quantizes at the ulp of the local field magnitude — the staircase
in the figure. Result: exact zeros through $t \approx 5.3$ (and intermittently
after), at most a few ulp (relative $< 10^{-6}$) through $t = 12.7$ (MG) / $13.2$
(FFT), then explosive growth as the past-ceiling runaway amplifies round-off into
$O(1)$ asymmetry — visible in the difference maps concentrating on the knots.

In [ ]:
## Q2.1 follow-up (ii) — measuring the parity conservation used throughout this
## section. The reflection is P: (x,y) → (−x,−y); on the cell-centered grid no cell
## sits at the origin, so the mirror of cell index i is mod1(257 − i, 256)
## (cell centers ±Δ/2, ±3Δ/2, … pair up exactly). Metric, per snapshot:
##     a(t) = max_{i,j} | S(i,j) − S(mirror(i), mirror(j)) |
## on the z-averaged density S. a = 0.0 means the two mirror halves of the stored
## field are BITWISE identical. Sweeps cache to run/q21_parity_{mg,fft}.txt
## (~2 min each at first run).
function parity_sweep(dir, tag)
    cf = joinpath(RUN, "q21_parity_$tag.txt")
    if !isfile(cf)
        fs = sort(filter(f -> startswith(f, "SwingSG.hydro_u_d"),
                         readdir(joinpath(dir, "bin"))))
        open(cf, "w") do io
            println(io, "# t  max|S - S∘P|  max(S)")
            for f in fs
                A = mosaic(joinpath(dir, "bin", f), "dens")
                S = dropdims(mean(A; dims=3); dims=3)
                a = maximum(abs(S[i, j] - S[mod1(257 - i, 256), mod1(257 - j, 256)])
                            for i in 1:256, j in 1:256)
                @printf(io, "%.4f  %.6e  %.6e\n",
                        0.05 * parse(Int, split(f, '.')[end-1]), a, maximum(S))
            end
        end
    end
    readcols(cf)
end
pa_mg = parity_sweep(D256MG, "mg")
pa_fft = parity_sweep(D256FFT, "fft")
nz(pa) = count(pa[:, 2] .== 0.0)
@printf("snapshots with a(t) = 0.0 EXACTLY (bitwise mirror halves): MG %d/%d, FFT %d/%d\n",
        nz(pa_mg), size(pa_mg, 1), nz(pa_fft), size(pa_fft, 1))
lastok(pa) = pa[findlast(pa[:, 2] ./ pa[:, 3] .< 1e-6), 1]
@printf("last snapshot with relative asymmetry < 1e-6: MG t = %.2f, FFT t = %.2f\n",
        lastok(pa_mg), lastok(pa_fft))

pmap(idx) = begin
    A = mosaic(joinpath(D256MG, "bin", @sprintf("SwingSG.hydro_u_d.%05d.bin", idx)), "dens")
    S = dropdims(mean(A; dims=3); dims=3)
    [abs(S[i, j] - S[mod1(257 - i, 256), mod1(257 - j, 256)]) for i in 1:256, j in 1:256]
end
fig = Figure(size=(1180, 740))
ax = Axis(fig[1, 1:2]; xlabel="t", ylabel="max |S − S∘P|", yscale=log10,
          title="parity asymmetry per snapshot (points on the bottom rail are EXACT zeros)")
for (pa, c, nm) in ((pa_mg, C1, "MG"), (pa_fft, C2, "FFT"))
    scatterlines!(ax, pa[:, 1], max.(pa[:, 2], 1e-9); color=c, markersize=4,
                  linewidth=0.8, label=nm)
end
hlines!(ax, [3.8e-6]; color=(:gray, 0.5), linestyle=:dot)
text!(ax, 0.2, 5e-6; text="one float32 ulp at ρ ≈ 40", fontsize=10, color=:gray40)
vlines!(ax, [12.55]; color=(:red, 0.4), linestyle=:dash)
text!(ax, 12.6, 1e-6; text="MG past Truelove", fontsize=10, color=(:red, 0.6))
axislegend(ax; position=:lt, framevisible=false)

xs256 = range(-π, π; length=256)
for (p, idx, ttl) in ((1, 250, "t = 12.5: |S − S∘P| (one ulp, at the knot)"),
                      (2, 280, "t = 14: symmetry broken by the runaway"))
    axm = Axis(fig[2, p]; xlabel="x", ylabel=p == 1 ? "y" : "", aspect=DataAspect(),
               title=ttl, titlesize=12)
    hm = heatmap!(axm, xs256, xs256, log10.(max.(pmap(idx), 1e-9)); colormap=:magma,
                  colorrange=(-9, 5))
    p == 2 && Colorbar(fig[2, 3], hm; label="log₁₀ |S − S∘P|")
end
fig

#### Closure — breaking the symmetry: clumps at generic sites

The remaining conviction test (requested 2026-08-19): perturb the IC so that *nothing*
is symmetry-pinned, and check that clumps then form away from the axes and all
boundaries. Design constraint from the shear: inversion symmetry of the shearing-box
dynamics holds only about centers with $x_c = 0$ (the background $v_y = -q\Omega x$ is
odd about $x=0$ alone), and a single cosine always has inversion centers on that axis
— so breaking parity requires a second, **non-parallel** wave with generic phase,
whose common inversion center with the primary sits at $x_c \ne 0$ and is destroyed by
the shear immediately. (A parallel second mode — the primary's own harmonic — would
instead leave a continuous along-crest degeneracy and a noise-selected site.)

The swing pgen gained `<problem> nwx2/nwy2/amp2/phase2` (commit `ff62f828`, defaults
bitwise-inert; `phase0` from the translation test is the same commit):
$$\rho = \rho_0\big[1 + 0.2\cos(-6x + y) + 0.1\cos(-11x + 2y + \psi)\big],$$
run at uniform $128^2$ with $\psi = 0.8$ and $2.3$ (`clump_2m_a/b`; both modes are
leading waves with swing times 4.0 / 3.67). The initial fields match the analytic
form to $6\times10^{-8}$ (float32 floor) — the cut figure below.

| run | final clump (t = 10) | peak $\rho$ |
|---|---|---|
| single mode | $(+0.025, -0.025)$ — the parity crosshair | 131 |
| two-mode $\psi = 0.8$ | $(+2.430, -0.368)$ | $6.5\times10^4$ |
| two-mode $\psi = 2.3$ | $(+1.301, +1.203)$ | $1.8\times10^4$ |

**Clumps at fully generic sites, tracking $\psi$** — far from $x=0$, $y \in \{0,\pi\}$,
and every boundary. Together with the translation test this closes the question from
both directions: when symmetry pins the site the clump lands exactly where symmetry
dictates; when nothing pins it, the clump lands where the physics puts it, with zero
attraction to grid-special locations. Movie:
`movie_runs/dens_clump_parity_breaking.mp4` (column-max projection — the deep cores
sit off the mid-plane, so a $z=0$ slice shows an evacuated void where the clump is).
Caveat: the two-mode peaks are far past the $128^2$ Truelove ceiling (171) —
positions meaningful, depths not.

*Planned: the same two runs at $256^2$ on the cluster; a comparison cell will be
added when that data arrives.*


In [ ]:
## Q2.1 parity-breaking test — initial density profiles (z = 0 mid-plane cuts).
## Original: rho = 1 + 0.2 cos(-6x + y). Two-mode: + 0.1 cos(-11x + 2y + psi),
## psi = 0.8 (run a) / 2.3 (run b). The second, non-parallel wave leaves no
## inversion center on the x = 0 axis, so the shear preserves no parity at all.
runs2m = (("clump_ph_0", "single mode (original)", 0.0, false, C1),
          ("clump_2m_a", "two-mode, ψ = 0.8", 0.8, true, C3),
          ("clump_2m_b", "two-mode, ψ = 2.3", 2.3, true, C2))
xs128 = [-π + (i - 0.5) * 2π / 128 for i in 1:128]
jc = 64                                  # row/column nearest 0 (coordinate -0.0245)
ana2m(x, y, ψ, two) = 1 + 0.2 * cos(-6x + y) + (two ? 0.1 * cos(-11x + 2y + ψ) : 0.0)

fig = Figure(size=(1080, 420))
ax1 = Axis(fig[1, 1]; xlabel="x", ylabel="ρ at z = 0",
           title="initial ρ(x) at y ≈ 0", xticks=([-π, 0, π], ["−π", "0", "π"]))
ax2 = Axis(fig[1, 2]; xlabel="y", title="initial ρ(y) at x ≈ 0",
           xticks=([-π, -π/2, 0, π/2, π], ["−π", "−π/2", "0", "π/2", "π"]))
merr = 0.0
for (bn, lb, ψ, two, c) in runs2m
    A = mosaic(joinpath(RUN, "bin", bn * ".hydro_u_d.00000.bin"), "dens")
    lines!(ax1, xs128, A[:, jc, 8]; color=c, linewidth=1.8, label=lb)
    lines!(ax2, xs128, A[jc, :, 8]; color=c, linewidth=1.8, label=lb)
    ax_ = [ana2m(x, xs128[jc], ψ, two) for x in xs128]
    ay_ = [ana2m(xs128[jc], y, ψ, two) for y in xs128]
    lines!(ax1, xs128, ax_; color=:black, linestyle=:dot, linewidth=0.8)
    lines!(ax2, xs128, ay_; color=:black, linestyle=:dot, linewidth=0.8)
    global merr = max(merr,
                      maximum(abs.(A[:, jc, 8] .- ax_)), maximum(abs.(A[jc, :, 8] .- ay_)))
end
axislegend(ax1; position=:rb, framevisible=false, labelsize=10)
axislegend(ax2; position=:rb, framevisible=false, labelsize=10)
Label(fig[0, 1:2],
      "t = 0 cuts: the second wave beats against the primary — no mirror symmetry, no special point on the grid (dots: analytic IC)";
      fontsize=13, tellwidth=false)
@printf("max |data − analytic| over all cuts: %.2e\n", merr)
fig

In [ ]:
## Parity-breaking census — final clump sites of the three 128^2 runs (column max
## over z of the last snapshot, t = 10).
co128(i) = -π + (i - 0.5) * 2π / 128
println("run           final max rho      site (x, y)        nearest special location")
for (bn, nm) in (("clump_ph_0", "single mode"), ("clump_2m_a", "two-mode ψ=0.8"),
                 ("clump_2m_b", "two-mode ψ=2.3"))
    A = mosaic(joinpath(RUN, "bin", bn * ".hydro_u_d.00040.bin"), "dens")
    M = dropdims(maximum(A; dims=3); dims=3)
    i = argmax(M)
    xc, yc = co128(i[1]), co128(i[2])
    dspec = min(abs(xc), π - abs(xc), abs(yc), π - abs(yc))
    @printf("%-14s %10.2f     (%+.3f, %+.3f)     %.2f box radians away\n",
            nm, M[i], xc, yc, dspec)
end
println("(the two-mode sites sit far from x=0, y=0, y=±π and every boundary, and move")
println(" with ψ — nothing in the code attracts clumps to grid-special locations.")
println(" Peak densities are far past the 128² Truelove ceiling of 171: positions are")
println(" meaningful, depths are not — the two-mode seeds are more concentrated and")
println(" collapse earlier, running away further by t = 10.)")

#### The $256^2$ cluster runs (2026-08-21)

The same two two-mode collapses at $256^2$ (NAS, 64 ranks; directories
`run/swing_mg_uniform_256_phase2_{0.8,2.3}`, basenames `clump_2m_{a,b}256`), against
the existing single-mode reference (`swing_mg_uniform_256_tlim14`). Two operational
lessons worth recording:
- **The first attempt ran single-mode**: the cluster input was pulled but the binary
  was stale, so the new `<problem>` parameters were silently ignored. The smoking gun
  in the job log was the warning `<problem>/phase0 ... was not used by the code`; the
  definitive check is frame 0 against the analytic IC (first cell below) — with the
  rebuilt binary it matches to the float32 floor.
- **The $\psi = 0.8$ run hit winner-takes-all early**: $4.3\times10^5\,\rho_0$
  already at $t = 9$ (630$\times$ past the $256^2$ Truelove ceiling, deeper than the
  reference's $t{=}14$ endpoint), so its timestep collapsed to a few $10^{-4}$ and
  the job crawled — it was stopped, and its useful span is $t \le 9$. This is the
  validation document's Truelove/sink lesson arriving in practice: past-ceiling
  evolution is unphysical *and* computationally unbounded. (The $\psi = 2.3$ run's
  runaway came later and shallower — $6.4\times10^4$ by $t{=}14$ — so it completed;
  note its `.hst` was restarted mid-run and needs `seg()`.)

Movie: `movie_runs/dens_clump_parity_breaking_256.mp4` ($t = 0 \to 14$; the
$\psi{=}0.8$ panel freezes at $t = 9$, the others play out).

In [ ]:
## 256^2 parity-breaking closure — initial density profiles (z = 0 mid-plane cuts)
## for the default seed (NAS SwingSG) and the two two-mode cluster runs, against
## the analytic ICs (parameter-free).
runs256 = ((joinpath(RUN, "swing_mg_uniform_256_tlim14"), "SwingSG",
            "single mode (default)", 0.0, false, C1),
           (joinpath(RUN, "swing_mg_uniform_256_phase2_0.8"), "clump_2m_a256",
            "two-mode, ψ = 0.8", 0.8, true, C3),
           (joinpath(RUN, "swing_mg_uniform_256_phase2_2.3"), "clump_2m_b256",
            "two-mode, ψ = 2.3", 2.3, true, C2))
xs256 = [-π + (i - 0.5) * 2π / 256 for i in 1:256]
jc256 = 128                              # row/column nearest 0 (coordinate -0.0123)
ana256(x, y, ψ, two) = 1 + 0.2 * cos(-6x + y) + (two ? 0.1 * cos(-11x + 2y + ψ) : 0.0)

fig = Figure(size=(1080, 420))
ax1 = Axis(fig[1, 1]; xlabel="x", ylabel="ρ at z = 0",
           title="initial ρ(x) at y ≈ 0", xticks=([-π, 0, π], ["−π", "0", "π"]))
ax2 = Axis(fig[1, 2]; xlabel="y", title="initial ρ(y) at x ≈ 0",
           xticks=([-π, -π/2, 0, π/2, π], ["−π", "−π/2", "0", "π/2", "π"]))
e256 = 0.0
for (dir, bn, lb, ψ, two, c) in runs256
    A = mosaic(joinpath(dir, "bin", bn * ".hydro_u_d.00000.bin"), "dens")
    lines!(ax1, xs256, A[:, jc256, 16]; color=c, linewidth=1.6, label=lb)
    lines!(ax2, xs256, A[jc256, :, 16]; color=c, linewidth=1.6, label=lb)
    ax_ = [ana256(x, xs256[jc256], ψ, two) for x in xs256]
    ay_ = [ana256(xs256[jc256], y, ψ, two) for y in xs256]
    lines!(ax1, xs256, ax_; color=:black, linestyle=:dot, linewidth=0.8)
    lines!(ax2, xs256, ay_; color=:black, linestyle=:dot, linewidth=0.8)
    global e256 = max(e256, maximum(abs.(A[:, jc256, 16] .- ax_)),
                      maximum(abs.(A[jc256, :, 16] .- ay_)))
end
axislegend(ax1; position=:rb, framevisible=false, labelsize=10)
axislegend(ax2; position=:rb, framevisible=false, labelsize=10)
Label(fig[0, 1:2],
      "256² cluster runs, t = 0 cuts (dots: analytic IC — the frame-0 check that caught the stale-binary first attempt)";
      fontsize=13, tellwidth=false)
@printf("max |data − analytic| over all six cuts: %.2e (float32 floor)\n", e256)
fig

In [ ]:
## 256^2 parity-breaking closure — density snapshots at t = 9 (the last time all
## three runs cover; the ψ=0.8 run's runaway Δt-crawl ends its useful span there).
## Column max over z, log scale, shared clipped range; census: top two knots per
## run (disk exclusion radius 1).
co256(i) = -π + (i - 0.5) * 2π / 256
colmax256(dir, bn, idx) = begin
    A = mosaic(joinpath(dir, "bin", @sprintf("%s.hydro_u_d.%05d.bin", bn, idx)), "dens")
    dropdims(maximum(A; dims=3); dims=3)
end
function top2(M; rex=41)                       # 41 cells ≈ 1 box radian
    i1 = argmax(M); Mm = copy(M)
    for di in -rex:rex, dj in -rex:rex
        di^2 + dj^2 <= rex^2 || continue
        Mm[mod1(i1[1] + di, 256), mod1(i1[2] + dj, 256)] = -Inf
    end
    i2 = argmax(Mm)
    (i1, M[i1], i2, Mm[i2])
end

snaps = ((joinpath(RUN, "swing_mg_uniform_256_tlim14"), "SwingSG", 180,
          "single mode, t = 9 (pre-collapse: onset t = 9.45)"),
         (joinpath(RUN, "swing_mg_uniform_256_phase2_0.8"), "clump_2m_a256", 36,
          "two-mode ψ = 0.8, t = 9"),
         (joinpath(RUN, "swing_mg_uniform_256_phase2_2.3"), "clump_2m_b256", 36,
          "two-mode ψ = 2.3, t = 9"))
xs9 = range(-π + π / 256, π - π / 256; length=256)
fig = Figure(size=(1500, 560))
hm9 = nothing
println("t = 9 census (column max; top two knots, exclusion radius 1):")
for (p, (dir, bn, idx, ttl)) in enumerate(snaps)
    M = colmax256(dir, bn, idx)
    ax = Axis(fig[1, p]; xlabel="x", ylabel=p == 1 ? "y" : "", title=ttl,
              titlesize=12, aspect=DataAspect())
    global hm9 = heatmap!(ax, xs9, xs9, log10.(max.(M, 1e-10)); colormap=:inferno,
                          colorrange=(-1.0, 2.6))
    i1, v1, i2, v2 = top2(M)
    if p > 1                                   # reference has no knots yet at t = 9
        for (i, v) in ((i1, v1), (i2, v2))
            scatter!(ax, [co256(i[1])], [co256(i[2])]; marker=:cross, color=:cyan,
                     markersize=14)
        end
    end
    @printf("%-28s knot %10.1f at (%+.3f, %+.3f)   2nd %10.1f at (%+.3f, %+.3f)\n",
            split(ttl, ',')[1], v1, co256(i1[1]), co256(i1[2]),
            v2, co256(i2[1]), co256(i2[2]))
end
Colorbar(fig[1, 4], hm9; label="log₁₀ max_z ρ/ρ₀ (clipped)")
println("(all two-mode knots at generic sites — compare the 128² single clumps at")
println(" (+2.430, -0.368) and (+1.301, +1.203): exact sites are resolution-sensitive,")
println(" as the slosh mechanism predicts, but no run puts a knot on an axis, a fixed")
println(" point, or a boundary)")
fig

**$256^2$ verdict.** Both two-mode runs fragment into *several* knots — a
richer census than the single $128^2$ clumps, as expected for a structured seed at
doubled resolution — and every knot sits at a fully generic site:

| run | dominant knot (t = 9) | second knot | 128² site (t = 10) |
|---|---|---|---|
| $\psi = 0.8$ | $4.3\times10^5$ at $(+0.70, +2.64)$ | $1.6\times10^4$ at $(-2.61, +0.36)$ | $(+2.43, -0.37)$ |
| $\psi = 2.3$ | $8.8\times10^4$ at $(-1.76, +1.56)$ | $1.1\times10^4$ at $(-2.91, -2.30)$ | $(+1.30, +1.20)$ |

The exact sites move between resolutions — exactly the sensitivity the slosh
mechanism predicts for unpinned collapse — but the *claim under test* holds
everywhere: **no run at any resolution places a knot on an axis, a parity fixed
point, or a boundary**, and the sites change with $\psi$. Meanwhile the single-mode
reference at $t=9$ is still a smooth pre-collapse wave (onset $t = 9.45$), after
which it collapses onto its symmetry-pinned wrap site as documented. The
parity-breaking closure holds at both resolutions.

#### Epilogue — the AMR stress test (2026-08-21)

Beyond the PI questions: the $\psi=2.3$ collapse rerun with density-triggered AMR
(input `swing_selfgrav_clump_amr256_2m.athinput`, commits `da8209e5` + `469d2a30`) —
the deepest exercise of the Phase-2 stack to date: AMR + FARGO annular rings +
shear-periodic multigrid at 64 ranks, chasing a genuinely nonlinear collapse.

**Attempt 1** (`value_max=10`, cap 160/rank; archived in `failed/`): the criterion
fired on the *swing transient* — the over-compressed crest exceeds $10\rho_0$ across
many columns at $t\approx4.4$, and the annular policy turned every flagged column
into full rings at every qualifying level: 10,144 blocks in the $t=4.5$ snapshot,
11,264 wanted at the fatal remesh. The cap guard aborted cleanly on all ranks; zero
multigrid warnings; history bitwise identical to the uniform twin until the first
remesh. Lesson: cap at the hard ceiling (512/rank $=$ whole box at level 2) and
refine on the clumps (`value_max=50`), not the bouncing transient.

**Attempt 2** (`value_max=50`, cap 512): the census below. Highlights — bitwise
identity with the uniform twin through $t=5.98$; a transient ring at $t=6.0$
(57.5 $>$ 50 for one output interval) created *and* derefined cleanly; from
$t=7.0$ nested level-1/2 rings lock onto the collapsing knots and follow them
(columns 8–11 $\to$ 6–12, up to $\sim$7,500 blocks); the dominant knot runs to
$5.1\times10^6\rho_0$ — a five-million-fold contrast, forty times deeper than
anything the program had reached — while every one of the 100+
$(\mathrm{level}, l_{x1}, l_{x3})$ rings stays complete and the boundary columns
stay untouched. The run ends in the CFL crawl ($\Delta t = 9.4\times10^{-6}$ at
$t=8.28$, implied flow speeds $\sim$200$c_s$ on the level-2 grid): the
Truelove/sink lesson in its sharpest form. Movie:
`movie_runs/dens_clump_amr256_stress.mp4` (uniform twin vs AMR, finest-grid
composite with block outlines).

In [ ]:
## Epilogue — the AMR stress test: per-snapshot census, ring invariant, twin check.
## Data: run/swing_mg_amr_256 (attempt 2: value_max=50, cap 512; attempt 1 in
## failed/). ~2 min: 34 multilevel snapshots.
using Statistics
DAMR = joinpath(RUN, "swing_mg_amr_256")
ha = read_hst(joinpath(DAMR, "clump_2m_b256amr.user.hst"))
hu2 = seg(read_hst(joinpath(RUN, "swing_mg_uniform_256_phase2_2.3", "clump_2m_b256.user.hst")))
nn = min(size(ha, 1), size(hu2, 1))
dv = abs.(ha[1:nn, 3] .- hu2[1:nn, 3])
i1 = findfirst(dv .> 0)
@printf("vs uniform twin: d_cos BITWISE identical through t = %.2f (first remesh %.2f)\n",
        ha[i1-1, 1], ha[i1, 1])
@printf("history ends t = %.2f with dt = %.1e (the runaway CFL crawl)\n\n",
        ha[end, 1], ha[end, 2])

println("t      blocks  L0    L1    L2   refined root cols     rings-incomplete   max rho")
NFR = length(filter(f -> startswith(f, "clump_2m_b256amr.hydro_u_d"),
                    readdir(joinpath(DAMR, "bin")))) - 1
for idx in 0:NFR
    fb = read_bin_blocks(joinpath(DAMR, "bin",
                                  @sprintf("clump_2m_b256amr.hydro_u_d.%05d.bin", idx)))
    lev = [b.logical[4] for b in fb.blocks]
    l0 = minimum(lev)
    nl = [count(lev .== l0 + k) for k in 0:2]
    bad = 0; cols = Int[]
    for k in 1:2
        bs = [b for b in fb.blocks if b.logical[4] == l0 + k]
        isempty(bs) && continue
        nyb = 16 * 2^k
        for key in unique((b.logical[1], b.logical[3]) for b in bs)
            count((b.logical[1], b.logical[3]) == key for b in bs) == nyb || (bad += 1)
        end
        append!(cols, [b.logical[1] >> k for b in bs])
    end
    iv = findfirst(==("dens"), fb.vars)
    mx = maximum(maximum(b.data[:, :, :, iv]) for b in fb.blocks)
    (idx <= 22 && nl[2] + nl[3] == 0 && idx % 4 != 0) && continue   # thin the root-only rows
    @printf("%5.2f  %6d  %4d %5d %5d   %-20s %3d              %10.1f\n", 0.25 * idx,
            length(fb.blocks), nl[1], nl[2], nl[3], string(sort(unique(cols))), bad, mx)
end
println("\n(rings-incomplete counts the TRUE policy invariant: full x2 coverage per")
println(" (level, lx1, lx3) PENCIL — partial-z refinement is legal, the FARGO shift")
println(" operates per (x,z) pencil. Zero everywhere = the annular policy held")
println(" through every remesh at up to ~7500 blocks.)")

In [ ]:
## Epilogue (continued) — the last synced snapshot, composited to the finest grid,
## refined MeshBlocks outlined.
let idx = 33
    fb = read_bin_blocks(joinpath(DAMR, "bin",
                                  @sprintf("clump_2m_b256amr.hydro_u_d.%05d.bin", idx)))
    iv = findfirst(==("dens"), fb.vars)
    l0 = minimum(b.logical[4] for b in fb.blocks)
    NFINE = 1024
    img = fill(-1.0f0, NFINE, NFINE)
    segs = Point2f[]
    dxf = 2π / NFINE
    for b in fb.blocks
        M = dropdims(maximum(b.data[:, :, :, iv]; dims=3); dims=3)
        n1, n2 = size(M)
        s = 2^(2 - (b.logical[4] - l0))
        i0 = round(Int, (b.geom[1] + π) / dxf); j0 = round(Int, (b.geom[3] + π) / dxf)
        for jj in 1:n2, ii in 1:n1
            v = log10(max(M[ii, jj], 1e-10))
            for a in 1:s, c in 1:s
                img[i0+(ii-1)*s+a, j0+(jj-1)*s+c] = max(img[i0+(ii-1)*s+a, j0+(jj-1)*s+c], v)
            end
        end
        b.logical[4] > l0 && push!(segs,
            Point2f(b.geom[1], b.geom[3]), Point2f(b.geom[2], b.geom[3]),
            Point2f(b.geom[2], b.geom[3]), Point2f(b.geom[2], b.geom[4]),
            Point2f(b.geom[2], b.geom[4]), Point2f(b.geom[1], b.geom[4]),
            Point2f(b.geom[1], b.geom[4]), Point2f(b.geom[1], b.geom[3]))
    end
    xs = range(-π + π / NFINE, π - π / NFINE; length=NFINE)
    global fig = Figure(size=(820, 700))
    ax = Axis(fig[1, 1]; xlabel="x", ylabel="y", aspect=DataAspect(),
              title=@sprintf("AMR stress test, t = %.2f: %d blocks, max ρ = %.2g (L2-resolved)",
                             0.25 * idx, length(fb.blocks),
                             maximum(maximum(b.data[:, :, :, iv]) for b in fb.blocks)))
    hm = heatmap!(ax, xs, xs, img; colormap=:inferno, colorrange=(-1.0f0, 3.0f0))
    linesegments!(ax, segs; color=(:lime, 0.45), linewidth=0.4)
    Colorbar(fig[1, 2], hm; label="log₁₀ max_z ρ/ρ₀ (clipped)")
end
fig

**Stress-test verdict.** Everything the Phase-2 machinery promises held at
scale: dozens of remeshes at thousands of blocks on 64 ranks, ring creation *and*
destruction, the shear-plane/octet rebuild at every event, the annular invariant
(full-$x_2$ per $(\mathrm{level}, l_{x1}, l_{x3})$ pencil — partial-$z$ refinement
is legal) unbroken throughout, boundary columns never refined, and the multigrid
solver clean through $5\times10^6$ density contrast. The one first-attempt failure
was the *configured cap*, aborting exactly as designed. Post-$t=6$ the AMR and
uniform histories diverge (the slosh sensitivity of Q2.1's follow-up — one
transient ring's truncation change reroutes the post-bounce path), so this is a
machinery test, not a site-by-site comparison; and as everywhere in this program,
past-ceiling densities are unphysical at every level — production runs stop or
sink at the ceiling.